# main arc

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import RobertaTokenizer, RobertaModel
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, Batch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from tqdm import tqdm
import warnings
import ast
warnings.filterwarnings('ignore')

import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"


class ICFGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_icfg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            func_calls = {}
            func_defs = {}
            
            def traverse(node, parent_id=None):
                nonlocal node_id
                current_id = node_id
                node_type = type(node).__name__
                nodes.append(node_type)
                node_id += 1
                
                if parent_id is not None:
                    edges.append([parent_id, current_id])
                
                if isinstance(node, ast.FunctionDef):
                    func_defs[node.name] = current_id
                
                if isinstance(node, ast.Call):
                    if isinstance(node.func, ast.Name):
                        func_name = node.func.id
                        if func_name not in func_calls:
                            func_calls[func_name] = []
                        func_calls[func_name].append(current_id)
                
                for child in ast.iter_child_nodes(node):
                    traverse(child, current_id)
            
            traverse(tree)
            
            for func_name, call_sites in func_calls.items():
                if func_name in func_defs:
                    def_id = func_defs[func_name]
                    for call_id in call_sites:
                        edges.append([call_id, def_id])
                        edges.append([def_id, call_id])
            
            if len(nodes) == 0:
                nodes = ['Module']
                edges = []
            
            return nodes, edges
        except:
            return ['Module'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            with torch.no_grad():
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class DFGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_dfg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            var_last_write = {}
            var_last_read = {}
            
            def extract_dataflow(node, current_id):
                if isinstance(node, ast.Name):
                    var_name = node.id
                    
                    if isinstance(node.ctx, ast.Store):
                        if var_name in var_last_read:
                            for read_id in var_last_read[var_name]:
                                edges.append([read_id, current_id])
                        var_last_write[var_name] = current_id
                        var_last_read[var_name] = []
                    
                    elif isinstance(node.ctx, ast.Load):
                        if var_name in var_last_write:
                            edges.append([var_last_write[var_name], current_id])
                        if var_name not in var_last_read:
                            var_last_read[var_name] = []
                        var_last_read[var_name].append(current_id)
                
                for child in ast.iter_child_nodes(node):
                    extract_dataflow(child, current_id)
            
            def traverse(node):
                nonlocal node_id
                current_id = node_id
                nodes.append(type(node).__name__)
                node_id += 1
                extract_dataflow(node, current_id)
                for child in ast.iter_child_nodes(node):
                    traverse(child)
            
            traverse(tree)
            
            if len(nodes) == 0:
                nodes = ['Program']
                edges = []
            
            return nodes, edges
        except:
            return ['Program'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            with torch.no_grad():
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class CDGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_cdg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            control_stack = []
            
            def traverse(node, parent_id=None):
                nonlocal node_id
                current_id = node_id
                node_type = type(node).__name__
                nodes.append(node_type)
                node_id += 1
                
                if parent_id is not None:
                    edges.append([parent_id, current_id])
                
                if isinstance(node, (ast.If, ast.For, ast.While, ast.Try)):
                    for ctrl_id in control_stack:
                        edges.append([ctrl_id, current_id])
                    control_stack.append(current_id)
                    
                    for child in ast.iter_child_nodes(node):
                        traverse(child, current_id)
                    
                    control_stack.pop()
                else:
                    if control_stack:
                        for ctrl_id in control_stack:
                            edges.append([ctrl_id, current_id])
                    
                    for child in ast.iter_child_nodes(node):
                        traverse(child, current_id)
            
            traverse(tree)
            
            if len(nodes) == 0:
                nodes = ['Program']
                edges = []
            
            return nodes, edges
        except:
            return ['Program'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            with torch.no_grad():
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class StableGraphEncoder(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=384, output_dim=512):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(0.2)
        self.residual = nn.Linear(input_dim, output_dim)
    
    def forward(self, x, edge_index):
        identity = self.residual(x)
        
        x = self.conv1(x, edge_index)
        x = self.norm1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        
        x = self.conv2(x, edge_index)
        x = self.norm2(x)
        
        x = x + identity
        return x


class HierarchicalGraphFusion(nn.Module):
    def __init__(self, graph_dim=512, num_heads=8):
        super().__init__()
        self.graph_dim = graph_dim
        
        self.icfg_encoder = StableGraphEncoder(768, 384, graph_dim)
        self.dfg_encoder = StableGraphEncoder(768, 384, graph_dim)
        self.cdg_encoder = StableGraphEncoder(768, 384, graph_dim)
        
        self.graph_attention = nn.MultiheadAttention(graph_dim, num_heads, dropout=0.1, batch_first=True)
        self.graph_norm = nn.LayerNorm(graph_dim)
        
        self.gate = nn.Sequential(
            nn.Linear(graph_dim * 3, graph_dim),
            nn.Sigmoid()
        )
        
        self.fusion = nn.Sequential(
            nn.Linear(graph_dim * 3, graph_dim * 2),
            nn.LayerNorm(graph_dim * 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(graph_dim * 2, graph_dim)
        )
        
    def forward(self, icfg_data, dfg_data, cdg_data):
        h_icfg = self.icfg_encoder(icfg_data.x, icfg_data.edge_index)
        h_dfg = self.dfg_encoder(dfg_data.x, dfg_data.edge_index)
        h_cdg = self.cdg_encoder(cdg_data.x, cdg_data.edge_index)
        
        icfg_global = torch.mean(h_icfg, dim=0, keepdim=True)
        dfg_global = torch.mean(h_dfg, dim=0, keepdim=True)
        cdg_global = torch.mean(h_cdg, dim=0, keepdim=True)
        
        graph_stack = torch.stack([icfg_global, dfg_global, cdg_global], dim=1)
        
        attended, _ = self.graph_attention(graph_stack, graph_stack, graph_stack)
        attended = self.graph_norm(attended + graph_stack)
        
        fused = torch.cat([attended[:, 0], attended[:, 1], attended[:, 2]], dim=-1)
        
        gate_weights = self.gate(fused)
        output = self.fusion(fused)
        output = output * gate_weights
        
        return output.squeeze(0)


class ImprovedUnifiedModel(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.tokenizer = RobertaTokenizer.from_pretrained('microsoft/graphcodebert-base')
        self.graphcodebert = RobertaModel.from_pretrained('microsoft/graphcodebert-base')
        
        for param in self.graphcodebert.parameters():
            param.requires_grad = True
        
        self.icfg_builder = ICFGBuilder(self.graphcodebert)
        self.dfg_builder = DFGBuilder(self.graphcodebert)
        self.cdg_builder = CDGBuilder(self.graphcodebert)
        
        self.graph_fusion = HierarchicalGraphFusion(graph_dim=512, num_heads=8)
        
        self.code_projection = nn.Linear(768, 512)
        
        self.multimodal_fusion = nn.Sequential(
            nn.Linear(512 + 512, 768),
            nn.LayerNorm(768),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(768, 512),
            nn.LayerNorm(512)
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
        self.criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
        
    def build_graph_data(self, code, device):
        icfg_nodes, icfg_edges = self.icfg_builder.build_icfg(code)
        dfg_nodes, dfg_edges = self.dfg_builder.build_dfg(code)
        cdg_nodes, cdg_edges = self.cdg_builder.build_cdg(code)
        
        icfg_x = self.icfg_builder.nodes_to_embeddings(icfg_nodes, self.tokenizer, device)
        dfg_x = self.dfg_builder.nodes_to_embeddings(dfg_nodes, self.tokenizer, device)
        cdg_x = self.cdg_builder.nodes_to_embeddings(cdg_nodes, self.tokenizer, device)
        
        if len(icfg_edges) == 0:
            icfg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
        else:
            icfg_edge_index = torch.tensor(icfg_edges, dtype=torch.long, device=device).t().contiguous()
        
        if len(dfg_edges) == 0:
            dfg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
        else:
            dfg_edge_index = torch.tensor(dfg_edges, dtype=torch.long, device=device).t().contiguous()
        
        if len(cdg_edges) == 0:
            cdg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
        else:
            cdg_edge_index = torch.tensor(cdg_edges, dtype=torch.long, device=device).t().contiguous()
        
        icfg_data = Data(x=icfg_x, edge_index=icfg_edge_index)
        dfg_data = Data(x=dfg_x, edge_index=dfg_edge_index)
        cdg_data = Data(x=cdg_x, edge_index=cdg_edge_index)
        
        return icfg_data, dfg_data, cdg_data
        
    def forward(self, code):
        device = next(self.parameters()).device
        
        icfg_data, dfg_data, cdg_data = self.build_graph_data(code, device)
        graph_repr = self.graph_fusion(icfg_data, dfg_data, cdg_data)
        
        tokens = self.tokenizer(
            code, 
            return_tensors='pt', 
            truncation=True, 
            max_length=512, 
            padding='max_length'
        )
        tokens = {k: v.to(device) for k, v in tokens.items()}
        
        code_output = self.graphcodebert(**tokens)
        code_repr = code_output.last_hidden_state[:, 0, :]
        code_repr = self.code_projection(code_repr)
        
        combined = torch.cat([graph_repr.unsqueeze(0), code_repr], dim=-1)
        fused_repr = self.multimodal_fusion(combined)
        
        logits = self.classifier(fused_repr)
        return logits


class CodeDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            'func': str(row['func']),
            'label': int(row['label'])
        }


def train_model(model, train_loader, val_loader, num_epochs=20, device='mps'):
    optimizer = torch.optim.AdamW([
        {'params': model.graphcodebert.parameters(), 'lr': 5e-6, 'weight_decay': 0.01},
        {'params': model.graph_fusion.parameters(), 'lr': 5e-5, 'weight_decay': 0.01},
        {'params': model.code_projection.parameters(), 'lr': 1e-4, 'weight_decay': 0.01},
        {'params': model.multimodal_fusion.parameters(), 'lr': 1e-4, 'weight_decay': 0.01},
        {'params': model.classifier.parameters(), 'lr': 1e-4, 'weight_decay': 0.01}
    ])
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=3, T_mult=2, eta_min=1e-7
    )
    
    best_val_f1 = 0
    patience = 8
    patience_counter = 0
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        train_preds = []
        train_labels = []
        
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
        for batch in progress_bar:
            code = batch['func'][0] if isinstance(batch['func'], list) else batch['func']
            label = batch['label']
            
            if isinstance(label, torch.Tensor):
                if label.dim() == 0:
                    label = label.item()
                else:
                    label = label[0].item() if len(label) > 0 else label.item()
            
            optimizer.zero_grad()
            
            logits = model(code)
            
            if logits.dim() == 1:
                logits = logits.unsqueeze(0)
            
            label_tensor = torch.tensor([label], dtype=torch.long, device=device)
            loss = model.criterion(logits, label_tensor)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            
            train_loss += loss.item()
            pred = torch.argmax(logits, dim=1).cpu().item()
            train_preds.append(pred)
            train_labels.append(label)
            
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        scheduler.step()
        
        train_acc = accuracy_score(train_labels, train_preds)
        _, _, train_f1, _ = precision_recall_fscore_support(
            train_labels, train_preds, average='weighted', zero_division=0
        )
        
        model.eval()
        val_preds = []
        val_labels = []
        
        with torch.no_grad():
            for batch in val_loader:
                code = batch['func'][0] if isinstance(batch['func'], list) else batch['func']
                label = batch['label']
                
                if isinstance(label, torch.Tensor):
                    if label.dim() == 0:
                        label = label.item()
                    else:
                        label = label[0].item() if len(label) > 0 else label.item()
                
                logits = model(code)
                
                if logits.dim() == 1:
                    logits = logits.unsqueeze(0)
                
                pred = torch.argmax(logits, dim=1).cpu().item()
                val_preds.append(pred)
                val_labels.append(label)
        
        val_acc = accuracy_score(val_labels, val_preds)
        _, _, val_f1, _ = precision_recall_fscore_support(
            val_labels, val_preds, average='weighted', zero_division=0
        )
        
        print(f'\nEpoch {epoch+1}/{num_epochs}:')
        print(f'  Train: Loss={train_loss/len(train_loader):.4f}, Acc={train_acc:.4f}, F1={train_f1:.4f}')
        print(f'  Val:   Acc={val_acc:.4f}, F1={val_f1:.4f}')
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pt')
            print(f'  ✓ Best model saved (F1: {best_val_f1:.4f})')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'\nEarly stopping at epoch {epoch+1}')
                break
    
    print(f'\nLoading best model (F1: {best_val_f1:.4f})')
    model.load_state_dict(torch.load('best_model.pt', map_location=device, weights_only=True))
    return model


def evaluate_model(model, test_loader, device='mps'):
    model.eval()
    test_preds = []
    test_labels = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Testing'):
            code = batch['func'][0] if isinstance(batch['func'], list) else batch['func']
            label = batch['label']
            
            if isinstance(label, torch.Tensor):
                if label.dim() == 0:
                    label = label.item()
                else:
                    label = label[0].item() if len(label) > 0 else label.item()
            
            logits = model(code)
            
            if logits.dim() == 1:
                logits = logits.unsqueeze(0)
            
            pred = torch.argmax(logits, dim=1).cpu().item()
            test_preds.append(pred)
            test_labels.append(label)
    
    test_acc = accuracy_score(test_labels, test_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        test_labels, test_preds, average='weighted', zero_division=0
    )
    
    print(f'\n{"="*70}')
    print(f'{"FINAL TEST RESULTS":^70}')
    print(f'{"="*70}')
    print(f'  Accuracy:  {test_acc:.4f}')
    print(f'  Precision: {precision:.4f}')
    print(f'  Recall:    {recall:.4f}')
    print(f'  F1 Score:  {f1:.4f}')
    print(f'{"="*70}\n')
    
    print('Classification Report:')
    print(classification_report(test_labels, test_preds, zero_division=0))
    
    return test_acc, precision, recall, f1


if __name__ == '__main__':
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f'Device: {device}')

    df = pd.read_csv('/Users/akter/fahim/data/trainpro (1).csv')

    
    print(f'\n{"="*70}')
    print('Dataset Information:')
    print(f'  Shape: {df.shape}')
    print(f'  Columns: {df.columns.tolist()}')
    print('\nLabel Distribution:')
    print(df["label"].value_counts().sort_index())
    
    train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])
    
    print(f'\nData Split:')
    print(f'  Train: {len(train_df)}')
    print(f'  Val:   {len(val_df)}')
    print(f'  Test:  {len(test_df)}')
    
    train_dataset = CodeDataset(train_df)
    val_dataset = CodeDataset(val_df)
    test_dataset = CodeDataset(test_df)
    
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=1, shuffle=False)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False)
    
    print('\nInitializing model...')
    model = ImprovedUnifiedModel(num_classes=6).to(device)
    
    print(f'  Parameters: {sum(p.numel() for p in model.parameters()):,}')
    
    print(f'\n{"="*70}')
    print('TRAINING')
    print(f'{"="*70}')
    model = train_model(model, train_loader, val_loader, num_epochs=20, device=device)
    
    print(f'\n{"="*70}')
    print('FINAL EVALUATION')
    print(f'{"="*70}')
    test_acc, precision, recall, f1 = evaluate_model(model, test_loader, device=device)
    
    torch.save(model.state_dict(), 'final_model1.pt')
    print(f'\nModel saved: final_model.pt')

# Traditional Classifer on Draper with codegraphnet extractors embed

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, AdamW
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error
)
import xgboost as xgb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
import warnings
warnings.filterwarnings('ignore')


def load_and_sample_data(train_path, test_path):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    train_label0 = train_df[train_df['label'] == 0].sample(n=3800, random_state=42)
    train_others = train_df[train_df['label'] != 0]
    train_df = pd.concat([train_label0, train_others], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
    
    test_label0 = test_df[test_df['label'] == 0].sample(n=900, random_state=42)
    test_others = test_df[test_df['label'] != 0]
    test_df = pd.concat([test_label0, test_others], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
    
    X_train = train_df.drop('label', axis=1).values
    y_train = train_df['label'].values
    X_test = test_df.drop('label', axis=1).values
    y_test = test_df['label'].values
    
    return X_train, y_train, X_test, y_test


def calculate_metrics(y_true, y_pred, y_pred_proba=None, num_classes=6):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    
    cm = confusion_matrix(y_true, y_pred)
    tp = np.diag(cm).sum()
    fn = cm.sum(axis=1).sum() - tp
    
    auc = 0.0
    if y_pred_proba is not None:
        try:
            auc = roc_auc_score(pd.get_dummies(y_true), y_pred_proba, multi_class='ovr', average='macro')
        except:
            auc = 0.0
    
    return {
        'AUC': auc,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'MCC': mcc,
        'Kappa': kappa,
        'MSE': mse,
        'MAE': mae,
        'TP': tp,
        'FN': fn
    }


def print_results(model_name, metrics):
    print(f'\n{"="*80}')
    print(f'{model_name:^80}')
    print(f'{"="*80}')
    print(f'  AUC:       {metrics["AUC"]:.4f}')
    print(f'  Accuracy:  {metrics["Accuracy"]:.4f}')
    print(f'  Precision: {metrics["Precision"]:.4f}')
    print(f'  Recall:    {metrics["Recall"]:.4f}')
    print(f'  F1 Score:  {metrics["F1"]:.4f}')
    print(f'  MCC:       {metrics["MCC"]:.4f}')
    print(f'  Kappa:     {metrics["Kappa"]:.4f}')
    print(f'  MSE:       {metrics["MSE"]:.4f}')
    print(f'  MAE:       {metrics["MAE"]:.4f}')
    print(f'  TP:        {metrics["TP"]}')
    print(f'  FN:        {metrics["FN"]}')
    print(f'{"="*80}\n')


def train_gru(X_train, y_train, X_test, y_test, num_classes):
    print('Training GRU...')
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    X_train_gru = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
    X_test_gru = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))
    
    model = Sequential()
    model.add(GRU(128, input_shape=(X_train_gru.shape[1], X_train_gru.shape[2]), return_sequences=True))
    model.add(Dropout(0.5))
    model.add(GRU(64))
    model.add(Dropout(0.5))
    model.add(Dense(num_classes, activation='softmax'))
    
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    model.fit(X_train_gru, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=0)
    
    y_pred_proba = model.predict(X_test_gru, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=-1)
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba, num_classes)
    print_results('GRU MODEL', metrics)
    return metrics


def train_xgboost(X_train, y_train, X_test, y_test, num_classes):
    print('Training XGBoost...')
    model = xgb.XGBClassifier(eval_metric='mlogloss', random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba, num_classes)
    print_results('XGBOOST MODEL', metrics)
    return metrics


def train_svm(X_train, y_train, X_test, y_test, num_classes):
    print('Training SVM...')
    model = SVC(kernel='linear', probability=True, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba, num_classes)
    print_results('SVM MODEL', metrics)
    return metrics


def train_deeptree(X_train, y_train, X_test, y_test, num_classes):
    print('Training DeepTree...')
    dt_classifier = DecisionTreeClassifier(random_state=42, max_depth=30)
    dt_classifier.fit(X_train, y_train)
    
    X_train_transformed = dt_classifier.predict_proba(X_train)
    X_test_transformed = dt_classifier.predict_proba(X_test)
    
    model = Sequential([
        Dense(256, activation='relu', input_shape=(X_train_transformed.shape[1],)),
        Dense(128, activation='relu'),
        Dense(64, activation='relu'),
        Dense(32, activation='relu'),
        Dense(16, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(X_train_transformed, y_train, epochs=5, batch_size=16, validation_split=0.2, verbose=0)
    
    y_pred_proba = model.predict(X_test_transformed, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba, num_classes)
    print_results('DEEPTREE MODEL', metrics)
    return metrics


def train_sgd(X_train, y_train, X_test, y_test, num_classes):
    print('Training SGD...')
    model = SGDClassifier(loss='hinge', penalty='l2', alpha=1000, learning_rate='constant',
                         eta0=0.0001, max_iter=65, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    metrics = calculate_metrics(y_test, y_pred, None, num_classes)
    print_results('SGD CLASSIFIER', metrics)
    return metrics


def train_decision_tree(X_train, y_train, X_test, y_test, num_classes):
    print('Training Decision Tree...')
    model = DecisionTreeClassifier(max_depth=30, min_samples_split=5, min_samples_leaf=2, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba, num_classes)
    print_results('DECISION TREE', metrics)
    return metrics


def train_random_forest(X_train, y_train, X_test, y_test, num_classes):
    print('Training Random Forest...')
    model = RandomForestClassifier(n_estimators=100, min_samples_split=2, max_depth=None,
                                  max_features='sqrt', random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba, num_classes)
    print_results('RANDOM FOREST', metrics)
    return metrics


def train_lstm(X_train, y_train, X_test, y_test, num_classes):
    print('Training LSTM...')
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
    y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)
    
    model = Sequential([
        Dense(256, activation='relu', input_shape=(X_train_scaled.shape[1],)),
        Dropout(0.5),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                 loss='categorical_crossentropy', metrics=['accuracy'])
    
    early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    model.fit(X_train_scaled, y_train_cat, epochs=50, batch_size=32,
             validation_split=0.2, callbacks=[early_stopping], verbose=0)
    
    y_pred_proba = model.predict(X_test_scaled, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba, num_classes)
    print_results('LSTM MODEL', metrics)
    return metrics


class NumericalDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return {'features': self.features[idx], 'labels': self.labels[idx]}


class NumericalTransformerClassifier(nn.Module):
    def __init__(self, input_dim, num_labels):
        super(NumericalTransformerClassifier, self).__init__()
        self.transformer = BertModel.from_pretrained('bert-base-uncased')
        self.embedding = nn.Linear(input_dim, self.transformer.config.hidden_size)
        self.dropout = nn.Dropout(0.5)
        self.classifier = nn.Linear(self.transformer.config.hidden_size, num_labels)

    def forward(self, features):
        embeddings = self.embedding(features)
        transformer_output = self.transformer(inputs_embeds=embeddings.unsqueeze(1)).last_hidden_state[:, 0, :]
        output = self.classifier(self.dropout(transformer_output))
        return output


def train_bert(X_train, y_train, X_test, y_test, num_classes):
    print('Training BERT...')
    train_dataset = NumericalDataset(X_train, y_train)
    test_dataset = NumericalDataset(X_test, y_test)
    
    batch_size = 64
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    input_dim = X_train.shape[1]
    model = NumericalTransformerClassifier(input_dim, num_classes)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=2e-6, weight_decay=0.001)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    epochs = 40
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            features = batch['features'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    model.eval()
    predictions, actuals, all_outputs_prob = [], [], []
    with torch.no_grad():
        for batch in test_loader:
            features = batch['features'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(features)
            
            outputs_prob = nn.functional.softmax(outputs, dim=1).cpu().numpy()
            all_outputs_prob.extend(outputs_prob)
            
            _, preds = torch.max(outputs, dim=1)
            predictions.extend(preds.cpu().numpy())
            actuals.extend(labels.cpu().numpy())
    
    y_pred = np.array(predictions)
    y_true = np.array(actuals)
    y_pred_proba = np.array(all_outputs_prob)
    
    metrics = calculate_metrics(y_true, y_pred, y_pred_proba, num_classes)
    print_results('BERT TRANSFORMER', metrics)
    return metrics


if __name__ == '__main__':
    train_path = '/Users/akter/fahim/codegraph/traintry1.csv'
    test_path = '/Users/akter/fahim/codegraph/testtry1.csv'
    
    print('Loading and sampling data...')
    X_train, y_train, X_test, y_test = load_and_sample_data(train_path, test_path)
    
    num_classes = len(np.unique(y_train))
    
    print(f'\nDataset Information:')
    print(f'  Training samples:   {X_train.shape[0]}')
    print(f'  Test samples:       {X_test.shape[0]}')
    print(f'  Feature dimensions: {X_train.shape[1]}')
    print(f'  Number of classes:  {num_classes}')
    
    print(f'\nTraining Label Distribution:')
    unique, counts = np.unique(y_train, return_counts=True)
    for label, count in zip(unique, counts):
        print(f'  Class {label}: {count}')
    
    print(f'\nTest Label Distribution:')
    unique, counts = np.unique(y_test, return_counts=True)
    for label, count in zip(unique, counts):
        print(f'  Class {label}: {count}')
    
    results = {}
    
    results['GRU'] = train_gru(X_train, y_train, X_test, y_test, num_classes)
    results['XGBoost'] = train_xgboost(X_train, y_train, X_test, y_test, num_classes)
    results['SVM'] = train_svm(X_train, y_train, X_test, y_test, num_classes)
    results['DeepTree'] = train_deeptree(X_train, y_train, X_test, y_test, num_classes)
    results['SGD'] = train_sgd(X_train, y_train, X_test, y_test, num_classes)
    results['Decision Tree'] = train_decision_tree(X_train, y_train, X_test, y_test, num_classes)
    results['Random Forest'] = train_random_forest(X_train, y_train, X_test, y_test, num_classes)
    results['LSTM'] = train_lstm(X_train, y_train, X_test, y_test, num_classes)
    results['BERT'] = train_bert(X_train, y_train, X_test, y_test, num_classes)
    
    print('\n' + '='*120)
    print(f'{"SUMMARY OF ALL MODELS":^120}')
    print('='*120)
    print(f'{"Model":<15} {"AUC":<8} {"Acc":<8} {"Pre":<8} {"Rec":<8} {"F1":<8} {"MCC":<8} {"Kappa":<8} {"MSE":<8} {"MAE":<8} {"TP":<6} {"FN":<6}')
    print('-'*120)
    
    for model_name, metrics in results.items():
        print(f'{model_name:<15} {metrics["AUC"]:<8.4f} {metrics["Accuracy"]:<8.4f} {metrics["Precision"]:<8.4f} '
              f'{metrics["Recall"]:<8.4f} {metrics["F1"]:<8.4f} {metrics["MCC"]:<8.4f} {metrics["Kappa"]:<8.4f} '
              f'{metrics["MSE"]:<8.4f} {metrics["MAE"]:<8.4f} {metrics["TP"]:<6} {metrics["FN"]:<6}')
    
    print('='*120)
    
    best_model = max(results.items(), key=lambda x: x[1]['F1'])
    print(f'\nBest Model: {best_model[0]} (F1 Score: {best_model[1]["F1"]:.4f})')

# main abaltion

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import RobertaTokenizer, RobertaModel
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, Batch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from tqdm import tqdm
import warnings
import ast
warnings.filterwarnings('ignore')

import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"


class ICFGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_icfg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            func_calls = {}
            func_defs = {}
            
            def traverse(node, parent_id=None):
                nonlocal node_id
                current_id = node_id
                node_type = type(node).__name__
                nodes.append(node_type)
                node_id += 1
                
                if parent_id is not None:
                    edges.append([parent_id, current_id])
                
                if isinstance(node, ast.FunctionDef):
                    func_defs[node.name] = current_id
                
                if isinstance(node, ast.Call):
                    if isinstance(node.func, ast.Name):
                        func_name = node.func.id
                        if func_name not in func_calls:
                            func_calls[func_name] = []
                        func_calls[func_name].append(current_id)
                
                for child in ast.iter_child_nodes(node):
                    traverse(child, current_id)
            
            traverse(tree)
            
            for func_name, call_sites in func_calls.items():
                if func_name in func_defs:
                    def_id = func_defs[func_name]
                    for call_id in call_sites:
                        edges.append([call_id, def_id])
                        edges.append([def_id, call_id])
            
            if len(nodes) == 0:
                nodes = ['Module']
                edges = []
            
            return nodes, edges
        except:
            return ['Module'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device, frozen=False):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            if frozen:
                with torch.no_grad():
                    outputs = self.graphcodebert(**tokens)
                    embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            else:
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class DFGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_dfg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            var_last_write = {}
            var_last_read = {}
            
            def extract_dataflow(node, current_id):
                if isinstance(node, ast.Name):
                    var_name = node.id
                    
                    if isinstance(node.ctx, ast.Store):
                        if var_name in var_last_read:
                            for read_id in var_last_read[var_name]:
                                edges.append([read_id, current_id])
                        var_last_write[var_name] = current_id
                        var_last_read[var_name] = []
                    
                    elif isinstance(node.ctx, ast.Load):
                        if var_name in var_last_write:
                            edges.append([var_last_write[var_name], current_id])
                        if var_name not in var_last_read:
                            var_last_read[var_name] = []
                        var_last_read[var_name].append(current_id)
                
                for child in ast.iter_child_nodes(node):
                    extract_dataflow(child, current_id)
            
            def traverse(node):
                nonlocal node_id
                current_id = node_id
                nodes.append(type(node).__name__)
                node_id += 1
                extract_dataflow(node, current_id)
                for child in ast.iter_child_nodes(node):
                    traverse(child)
            
            traverse(tree)
            
            if len(nodes) == 0:
                nodes = ['Program']
                edges = []
            
            return nodes, edges
        except:
            return ['Program'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device, frozen=False):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            if frozen:
                with torch.no_grad():
                    outputs = self.graphcodebert(**tokens)
                    embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            else:
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class CDGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_cdg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            control_stack = []
            
            def traverse(node, parent_id=None):
                nonlocal node_id
                current_id = node_id
                node_type = type(node).__name__
                nodes.append(node_type)
                node_id += 1
                
                if parent_id is not None:
                    edges.append([parent_id, current_id])
                
                if isinstance(node, (ast.If, ast.For, ast.While, ast.Try)):
                    for ctrl_id in control_stack:
                        edges.append([ctrl_id, current_id])
                    control_stack.append(current_id)
                    
                    for child in ast.iter_child_nodes(node):
                        traverse(child, current_id)
                    
                    control_stack.pop()
                else:
                    if control_stack:
                        for ctrl_id in control_stack:
                            edges.append([ctrl_id, current_id])
                    
                    for child in ast.iter_child_nodes(node):
                        traverse(child, current_id)
            
            traverse(tree)
            
            if len(nodes) == 0:
                nodes = ['Program']
                edges = []
            
            return nodes, edges
        except:
            return ['Program'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device, frozen=False):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            if frozen:
                with torch.no_grad():
                    outputs = self.graphcodebert(**tokens)
                    embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            else:
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class ASTBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_ast(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            
            def traverse(node, parent_id=None):
                nonlocal node_id
                current_id = node_id
                node_type = type(node).__name__
                nodes.append(node_type)
                node_id += 1
                
                if parent_id is not None:
                    edges.append([parent_id, current_id])
                
                for child in ast.iter_child_nodes(node):
                    traverse(child, current_id)
            
            traverse(tree)
            
            if len(nodes) == 0:
                nodes = ['Module']
                edges = []
            
            return nodes, edges
        except:
            return ['Module'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device, frozen=False):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            if frozen:
                with torch.no_grad():
                    outputs = self.graphcodebert(**tokens)
                    embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            else:
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class CFGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_cfg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            
            def traverse(node, prev_id=None):
                nonlocal node_id
                current_id = node_id
                node_type = type(node).__name__
                nodes.append(node_type)
                node_id += 1
                
                if prev_id is not None:
                    edges.append([prev_id, current_id])
                
                if isinstance(node, ast.If):
                    test_id = current_id
                    body_start = node_id
                    for child in node.body:
                        traverse(child, test_id)
                        test_id = node_id - 1
                    
                    orelse_start = node_id
                    for child in node.orelse:
                        traverse(child, current_id)
                    return
                
                elif isinstance(node, (ast.For, ast.While)):
                    loop_id = current_id
                    for child in ast.iter_child_nodes(node):
                        traverse(child, loop_id)
                        loop_id = node_id - 1
                    edges.append([node_id - 1, current_id])
                    return
                
                else:
                    prev = current_id
                    for child in ast.iter_child_nodes(node):
                        traverse(child, prev)
                        prev = node_id - 1
            
            traverse(tree)
            
            if len(nodes) == 0:
                nodes = ['Module']
                edges = []
            
            return nodes, edges
        except:
            return ['Module'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device, frozen=False):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            if frozen:
                with torch.no_grad():
                    outputs = self.graphcodebert(**tokens)
                    embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            else:
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class StableGraphEncoder(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=384, output_dim=512, num_layers=2):
        super().__init__()
        self.num_layers = num_layers
        
        if num_layers == 1:
            self.conv1 = GCNConv(input_dim, output_dim)
            self.norm1 = nn.LayerNorm(output_dim)
            self.residual = nn.Linear(input_dim, output_dim)
        elif num_layers == 2:
            self.conv1 = GCNConv(input_dim, hidden_dim)
            self.conv2 = GCNConv(hidden_dim, output_dim)
            self.norm1 = nn.LayerNorm(hidden_dim)
            self.norm2 = nn.LayerNorm(output_dim)
            self.residual = nn.Linear(input_dim, output_dim)
        elif num_layers == 3:
            self.conv1 = GCNConv(input_dim, hidden_dim)
            self.conv2 = GCNConv(hidden_dim, hidden_dim)
            self.conv3 = GCNConv(hidden_dim, output_dim)
            self.norm1 = nn.LayerNorm(hidden_dim)
            self.norm2 = nn.LayerNorm(hidden_dim)
            self.norm3 = nn.LayerNorm(output_dim)
            self.residual = nn.Linear(input_dim, output_dim)
        
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x, edge_index):
        identity = self.residual(x)
        
        if self.num_layers == 1:
            x = self.conv1(x, edge_index)
            x = self.norm1(x)
            x = x + identity
        elif self.num_layers == 2:
            x = self.conv1(x, edge_index)
            x = self.norm1(x)
            x = F.gelu(x)
            x = self.dropout(x)
            x = self.conv2(x, edge_index)
            x = self.norm2(x)
            x = x + identity
        elif self.num_layers == 3:
            x = self.conv1(x, edge_index)
            x = self.norm1(x)
            x = F.gelu(x)
            x = self.dropout(x)
            x = self.conv2(x, edge_index)
            x = self.norm2(x)
            x = F.gelu(x)
            x = self.dropout(x)
            x = self.conv3(x, edge_index)
            x = self.norm3(x)
            x = x + identity
        
        return x


class HierarchicalGraphFusion(nn.Module):
    def __init__(self, graph_dim=512, num_heads=8, num_layers=2, num_graphs=3):
        super().__init__()
        self.graph_dim = graph_dim
        self.num_graphs = num_graphs
        
        self.icfg_encoder = StableGraphEncoder(768, 384, graph_dim, num_layers=num_layers)
        self.dfg_encoder = StableGraphEncoder(768, 384, graph_dim, num_layers=num_layers)
        self.cdg_encoder = StableGraphEncoder(768, 384, graph_dim, num_layers=num_layers)
        
        self.graph_attention = nn.MultiheadAttention(graph_dim, num_heads, dropout=0.1, batch_first=True)
        self.graph_norm = nn.LayerNorm(graph_dim)
        
        self.gate = nn.Sequential(
            nn.Linear(graph_dim * num_graphs, graph_dim),
            nn.Sigmoid()
        )
        
        self.fusion = nn.Sequential(
            nn.Linear(graph_dim * num_graphs, graph_dim * 2),
            nn.LayerNorm(graph_dim * 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(graph_dim * 2, graph_dim)
        )
        
    def forward(self, icfg_data, dfg_data, cdg_data):
        graphs_global = []
        
        if icfg_data is not None:
            h_icfg = self.icfg_encoder(icfg_data.x, icfg_data.edge_index)
            icfg_global = torch.mean(h_icfg, dim=0, keepdim=True)
            graphs_global.append(icfg_global)
        
        if dfg_data is not None:
            h_dfg = self.dfg_encoder(dfg_data.x, dfg_data.edge_index)
            dfg_global = torch.mean(h_dfg, dim=0, keepdim=True)
            graphs_global.append(dfg_global)
        
        if cdg_data is not None:
            h_cdg = self.cdg_encoder(cdg_data.x, cdg_data.edge_index)
            cdg_global = torch.mean(h_cdg, dim=0, keepdim=True)
            graphs_global.append(cdg_global)
        
        graph_stack = torch.stack(graphs_global, dim=1)
        
        attended, _ = self.graph_attention(graph_stack, graph_stack, graph_stack)
        attended = self.graph_norm(attended + graph_stack)
        
        fused = torch.cat([attended[:, i] for i in range(len(graphs_global))], dim=-1)
        
        gate_weights = self.gate(fused)
        output = self.fusion(fused)
        output = output * gate_weights
        
        return output.squeeze(0)


class AblationModel(nn.Module):
    def __init__(self, config, num_classes=6):
        super().__init__()
        self.config = config
        self.tokenizer = RobertaTokenizer.from_pretrained('microsoft/graphcodebert-base')
        self.graphcodebert = RobertaModel.from_pretrained('microsoft/graphcodebert-base')
        
        if config.get('finetune_bert', False):
            for param in self.graphcodebert.parameters():
                param.requires_grad = True
        else:
            for param in self.graphcodebert.parameters():
                param.requires_grad = False
        
        self.icfg_builder = ICFGBuilder(self.graphcodebert)
        self.dfg_builder = DFGBuilder(self.graphcodebert)
        self.cdg_builder = CDGBuilder(self.graphcodebert)
        self.ast_builder = ASTBuilder(self.graphcodebert)
        self.cfg_builder = CFGBuilder(self.graphcodebert)
        
        num_graphs = 0
        if config.get('use_icfg', False): num_graphs += 1
        if config.get('use_dfg', False): num_graphs += 1
        if config.get('use_cdg', False): num_graphs += 1
        if config.get('use_ast', False): num_graphs += 1
        if config.get('use_cfg', False): num_graphs += 1
        
        if config.get('use_graphs', False) and num_graphs > 0:
            gcn_layers = config.get('gcn_layers', 2)
            self.graph_fusion = HierarchicalGraphFusion(
                graph_dim=512, 
                num_heads=8, 
                num_layers=gcn_layers,
                num_graphs=num_graphs
            )
        
        self.code_projection = nn.Linear(768, 512)
        
        if config.get('use_graphs', False) and num_graphs > 0:
            if config.get('use_fusion', False):
                self.multimodal_fusion = nn.Sequential(
                    nn.Linear(512 + 512, 768),
                    nn.LayerNorm(768),
                    nn.GELU(),
                    nn.Dropout(0.2),
                    nn.Linear(768, 512),
                    nn.LayerNorm(512)
                )
                classifier_input = 512
            else:
                classifier_input = 512 + 512
        else:
            classifier_input = 512
        
        if not config.get('use_code_text', True):
            if config.get('use_graphs', False) and num_graphs > 0:
                classifier_input = 512
            else:
                classifier_input = 768
        
        self.classifier = nn.Sequential(
            nn.Linear(classifier_input, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
        self.criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
        
    def build_graph_data(self, code, device):
        frozen_embeddings = self.config.get('frozen_graph_embeddings', False)
        
        graphs = {}
        
        if self.config.get('use_icfg', False):
            icfg_nodes, icfg_edges = self.icfg_builder.build_icfg(code)
            icfg_x = self.icfg_builder.nodes_to_embeddings(icfg_nodes, self.tokenizer, device, frozen=frozen_embeddings)
            if len(icfg_edges) == 0:
                icfg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
            else:
                icfg_edge_index = torch.tensor(icfg_edges, dtype=torch.long, device=device).t().contiguous()
            graphs['icfg'] = Data(x=icfg_x, edge_index=icfg_edge_index)
        
        if self.config.get('use_dfg', False):
            dfg_nodes, dfg_edges = self.dfg_builder.build_dfg(code)
            dfg_x = self.dfg_builder.nodes_to_embeddings(dfg_nodes, self.tokenizer, device, frozen=frozen_embeddings)
            if len(dfg_edges) == 0:
                dfg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
            else:
                dfg_edge_index = torch.tensor(dfg_edges, dtype=torch.long, device=device).t().contiguous()
            graphs['dfg'] = Data(x=dfg_x, edge_index=dfg_edge_index)
        
        if self.config.get('use_cdg', False):
            cdg_nodes, cdg_edges = self.cdg_builder.build_cdg(code)
            cdg_x = self.cdg_builder.nodes_to_embeddings(cdg_nodes, self.tokenizer, device, frozen=frozen_embeddings)
            if len(cdg_edges) == 0:
                cdg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
            else:
                cdg_edge_index = torch.tensor(cdg_edges, dtype=torch.long, device=device).t().contiguous()
            graphs['cdg'] = Data(x=cdg_x, edge_index=cdg_edge_index)
        
        if self.config.get('use_ast', False):
            ast_nodes, ast_edges = self.ast_builder.build_ast(code)
            ast_x = self.ast_builder.nodes_to_embeddings(ast_nodes, self.tokenizer, device, frozen=frozen_embeddings)
            if len(ast_edges) == 0:
                ast_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
            else:
                ast_edge_index = torch.tensor(ast_edges, dtype=torch.long, device=device).t().contiguous()
            graphs['ast'] = Data(x=ast_x, edge_index=ast_edge_index)
        
        if self.config.get('use_cfg', False):
            cfg_nodes, cfg_edges = self.cfg_builder.build_cfg(code)
            cfg_x = self.cfg_builder.nodes_to_embeddings(cfg_nodes, self.tokenizer, device, frozen=frozen_embeddings)
            if len(cfg_edges) == 0:
                cfg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
            else:
                cfg_edge_index = torch.tensor(cfg_edges, dtype=torch.long, device=device).t().contiguous()
            graphs['cfg'] = Data(x=cfg_x, edge_index=cfg_edge_index)
        
        return graphs
        
    def forward(self, code):
        device = next(self.parameters()).device
        
        representations = []
        
        if self.config.get('use_graphs', False):
            graphs = self.build_graph_data(code, device)
            if len(graphs) > 0:
                icfg_data = graphs.get('icfg', None) if self.config.get('use_icfg', False) else None
                dfg_data = graphs.get('dfg', None) if self.config.get('use_dfg', False) else None
                cdg_data = graphs.get('cdg', None) if self.config.get('use_cdg', False) else None
                
                if self.config.get('use_ast', False):
                    icfg_data = graphs.get('ast', None)
                if self.config.get('use_cfg', False):
                    dfg_data = graphs.get('cfg', None)
                    cdg_data = None
                
                graph_repr = self.graph_fusion(icfg_data, dfg_data, cdg_data)
                representations.append(graph_repr.unsqueeze(0))
        
        if self.config.get('use_code_text', True):
            tokens = self.tokenizer(
                code, 
                return_tensors='pt', 
                truncation=True, 
                max_length=512, 
                padding='max_length'
            )
            tokens = {k: v.to(device) for k, v in tokens.items()}
            
            code_output = self.graphcodebert(**tokens)
            code_repr = code_output.last_hidden_state[:, 0, :]
            code_repr = self.code_projection(code_repr)
            representations.append(code_repr)
        
        if len(representations) == 0:
            tokens = self.tokenizer(
                code, 
                return_tensors='pt', 
                truncation=True, 
                max_length=512, 
                padding='max_length'
            )
            tokens = {k: v.to(device) for k, v in tokens.items()}
            code_output = self.graphcodebert(**tokens)
            fused_repr = code_output.last_hidden_state[:, 0, :]
        elif len(representations) == 1:
            fused_repr = representations[0]
        else:
            if self.config.get('use_fusion', False):
                combined = torch.cat(representations, dim=-1)
                fused_repr = self.multimodal_fusion(combined)
            else:
                fused_repr = torch.cat(representations, dim=-1)
        
        logits = self.classifier(fused_repr)
        return logits


class CodeDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            'func': str(row['func']),
            'label': int(row['label'])
        }


def train_model(model, train_loader, val_loader, num_epochs=20, device='mps', model_name='model'):
    optimizer = torch.optim.AdamW([
        {'params': model.graphcodebert.parameters(), 'lr': 5e-6, 'weight_decay': 0.01},
        {'params': [p for n, p in model.named_parameters() if 'graphcodebert' not in n], 'lr': 1e-4, 'weight_decay': 0.01}
    ])
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=3, T_mult=2, eta_min=1e-7
    )
    
    best_val_f1 = 0
    patience = 8
    patience_counter = 0
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        train_preds = []
        train_labels = []
        
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
        for batch in progress_bar:
            code = batch['func'][0] if isinstance(batch['func'], list) else batch['func']
            label = batch['label']
            
            if isinstance(label, torch.Tensor):
                if label.dim() == 0:
                    label = label.item()
                else:
                    label = label[0].item() if len(label) > 0 else label.item()
            
            optimizer.zero_grad()
            
            logits = model(code)
            
            if logits.dim() == 1:
                logits = logits.unsqueeze(0)
            
            label_tensor = torch.tensor([label], dtype=torch.long, device=device)
            loss = model.criterion(logits, label_tensor)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            
            train_loss += loss.item()
            pred = torch.argmax(logits, dim=1).cpu().item()
            train_preds.append(pred)
            train_labels.append(label)
            
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        scheduler.step()
        
        train_acc = accuracy_score(train_labels, train_preds)
        _, _, train_f1, _ = precision_recall_fscore_support(
            train_labels, train_preds, average='weighted', zero_division=0
        )
        
        model.eval()
        val_preds = []
        val_labels = []
        
        with torch.no_grad():
            for batch in val_loader:
                code = batch['func'][0] if isinstance(batch['func'], list) else batch['func']
                label = batch['label']
                
                if isinstance(label, torch.Tensor):
                    if label.dim() == 0:
                        label = label.item()

                    else:
                        label = label[0].item() if len(label) > 0 else label.item()
                
                logits = model(code)
                
                if logits.dim() == 1:
                    logits = logits.unsqueeze(0)
                
                pred = torch.argmax(logits, dim=1).cpu().item()
                val_preds.append(pred)
                val_labels.append(label)
        
        val_acc = accuracy_score(val_labels, val_preds)
        _, _, val_f1, _ = precision_recall_fscore_support(
            val_labels, val_preds, average='weighted', zero_division=0
        )
        
        print(f'\nEpoch {epoch+1}/{num_epochs}:')
        print(f'  Train: Loss={train_loss/len(train_loader):.4f}, Acc={train_acc:.4f}, F1={train_f1:.4f}')
        print(f'  Val:   Acc={val_acc:.4f}, F1={val_f1:.4f}')
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_counter = 0
            torch.save(model.state_dict(), f'best_{model_name}.pt')
            print(f'  ✓ Best model saved (F1: {best_val_f1:.4f})')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'\nEarly stopping at epoch {epoch+1}')
                break
    
    print(f'\nLoading best model (F1: {best_val_f1:.4f})')
    model.load_state_dict(torch.load(f'best_{model_name}.pt', map_location=device, weights_only=True))
    return model


def evaluate_model(model, test_loader, device='mps'):
    model.eval()
    test_preds = []
    test_labels = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Testing'):
            code = batch['func'][0] if isinstance(batch['func'], list) else batch['func']
            label = batch['label']
            
            if isinstance(label, torch.Tensor):
                if label.dim() == 0:
                    label = label.item()
                else:
                    label = label[0].item() if len(label) > 0 else label.item()
            
            logits = model(code)
            
            if logits.dim() == 1:
                logits = logits.unsqueeze(0)
            
            pred = torch.argmax(logits, dim=1).cpu().item()
            test_preds.append(pred)
            test_labels.append(label)
    
    test_acc = accuracy_score(test_labels, test_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        test_labels, test_preds, average='weighted', zero_division=0
    )
    
    return test_acc, precision, recall, f1


if __name__ == '__main__':
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f'Device: {device}')

    df = pd.read_csv('/Users/akter/fahim/data/trainpro (1).csv')
    train_label0_sample = df[df['label'] == 0].sample(n=3800, random_state=42)
    train_others = df[df['label'] != 0]
    df = pd.concat([train_label0_sample, train_others], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f'\n{"="*70}')
    print('Dataset Information:')
    print(f'  Shape: {df.shape}')
    print(f'  Columns: {df.columns.tolist()}')
    print('\nLabel Distribution:')
    print(df["label"].value_counts().sort_index())
    
    train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])
    
    print(f'\nData Split:')
    print(f'  Train: {len(train_df)}')
    print(f'  Val:   {len(val_df)}')
    print(f'  Test:  {len(test_df)}')
    
    train_dataset = CodeDataset(train_df)
    val_dataset = CodeDataset(val_df)
    test_dataset = CodeDataset(test_df)
    
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=8, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=8, shuffle=False)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=8, shuffle=False)
    
    configs = [
        # {
        #     'name': 'GraphCodeBERT',
        #     'use_graphs': False,
        #     'use_fusion': False,
        #     'finetune_bert': False,
        #     'use_code_text': True,
        #     'use_icfg': False,
        #     'use_dfg': False,
        #     'use_cdg': False,
        #     'use_ast': False,
        #     'use_cfg': False,
        #     'gcn_layers': 2,
        #     'frozen_graph_embeddings': False
        # },
        # {
        #     'name': 'GCB+Finetune',
        #     'use_graphs': False,
        #     'use_fusion': False,
        #     'finetune_bert': True,
        #     'use_code_text': True,
        #     'use_icfg': False,
        #     'use_dfg': False,
        #     'use_cdg': False,
        #     'use_ast': False,
        #     'use_cfg': False,
        #     'gcn_layers': 2,
        #     'frozen_graph_embeddings': False
        # },
        # {
        #     'name': 'GCB+Graphs',
        #     'use_graphs': True,
        #     'use_fusion': False,
        #     'finetune_bert': False,
        #     'use_code_text': True,
        #     'use_icfg': True,
        #     'use_dfg': True,
        #     'use_cdg': True,
        #     'use_ast': False,
        #     'use_cfg': False,
        #     'gcn_layers': 2,
        #     'frozen_graph_embeddings': False
        # },
        # {
        #     'name': 'GCB+Graphs+FT',
        #     'use_graphs': True,
        #     'use_fusion': False,
        #     'finetune_bert': True,
        #     'use_code_text': True,
        #     'use_icfg': True,
        #     'use_dfg': True,
        #     'use_cdg': True,
        #     'use_ast': False,
        #     'use_cfg': False,
        #     'gcn_layers': 2,
        #     'frozen_graph_embeddings': False
        # },
        # {
        #     'name': 'GCB+Fusion',
        #     'use_graphs': False,
        #     'use_fusion': True,
        #     'finetune_bert': False,
        #     'use_code_text': True,
        #     'use_icfg': False,
        #     'use_dfg': False,
        #     'use_cdg': False,
        #     'use_ast': False,
        #     'use_cfg': False,
        #     'gcn_layers': 2,
        #     'frozen_graph_embeddings': False
        # },
        {
            'name': 'Full(Proposed)',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': True,
            'use_dfg': True,
            'use_cdg': True,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 2,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'Full-ICFG',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': False,
            'use_dfg': True,
            'use_cdg': True,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 2,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'Full-DFG',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': True,
            'use_dfg': False,
            'use_cdg': True,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 2,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'Full-CDG',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': True,
            'use_dfg': True,
            'use_cdg': False,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 2,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'ICFG+DFG',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': True,
            'use_dfg': True,
            'use_cdg': False,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 2,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'ICFG+CDG',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': True,
            'use_dfg': False,
            'use_cdg': True,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 2,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'DFG+CDG',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': False,
            'use_dfg': True,
            'use_cdg': True,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 2,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'Full+AST',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': False,
            'use_dfg': False,
            'use_cdg': False,
            'use_ast': True,
            'use_cfg': False,
            'gcn_layers': 2,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'Full+CFG',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': False,
            'use_dfg': False,
            'use_cdg': False,
            'use_ast': False,
            'use_cfg': True,
            'gcn_layers': 2,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'Full+1GCN',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': True,
            'use_dfg': True,
            'use_cdg': True,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 1,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'Full+3GCN',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': True,
            'use_dfg': True,
            'use_cdg': True,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 3,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'Full-CodeText',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': False,
            'use_icfg': True,
            'use_dfg': True,
            'use_cdg': True,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 2,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'Full+FrozenEmbed',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': True,
            'use_dfg': True,
            'use_cdg': True,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 2,
            'frozen_graph_embeddings': True
        }
    ]
    
    results = []
    
    for config in configs:
        print(f'\n{"="*70}')
        print(f'CONFIGURATION: {config["name"]}')
        print(f'{"="*70}')
        print(f'Config: {config}')
        
        model = AblationModel(config, num_classes=6).to(device)
        print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
        
        model = train_model(model, train_loader, val_loader, num_epochs=5, device=device, model_name=config['name'])
        
        test_acc, precision, recall, f1 = evaluate_model(model, test_loader, device=device)
        
        results.append({
            'Configuration': config['name'],
            'Accuracy': test_acc,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1
        })
        
        print(f'\n{config["name"]} Results:')
        print(f'  Accuracy:  {test_acc:.4f}')
        print(f'  Precision: {precision:.4f}')
        print(f'  Recall:    {recall:.4f}')
        print(f'  F1 Score:  {f1:.4f}')
        
        torch.save(model.state_dict(), f'final_{config["name"]}.pt')
    
    results_df = pd.DataFrame(results)
    results_df.to_csv('ablation_results.csv', index=False)
    
    print(f'\n{"="*70}')
    print('ABLATION STUDY SUMMARY')
    print(f'{"="*70}')
    print(results_df.to_string(index=False))
    print(f'\nResults saved to: ablation_results.csv')

Device: mps

Dataset Information:
  Shape: (21793, 2)
  Columns: ['func', 'label']

Label Distribution:
label
0    3800
1    3566
2    3640
3    3575
4    3591
5    3621
Name: count, dtype: int64

Data Split:
  Train: 15255
  Val:   3269
  Test:  3269

CONFIGURATION: Full(Proposed)
Config: {'name': 'Full(Proposed)', 'use_graphs': True, 'use_fusion': True, 'finetune_bert': True, 'use_code_text': True, 'use_icfg': True, 'use_dfg': True, 'use_cdg': True, 'use_ast': False, 'use_cfg': False, 'gcn_layers': 2, 'frozen_graph_embeddings': False}


Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Parameters: 132,959,366


Epoch 1/5: 100%|███████████████| 1907/1907 [11:45<00:00,  2.70it/s, loss=2.4021]



Epoch 1/5:
  Train: Loss=1.3610, Acc=0.4971, F1=0.5040
  Val:   Acc=0.6161, F1=0.6275
  ✓ Best model saved (F1: 0.6275)


...


Epoch 4/5:
  Train: Loss=1.1822, Acc=0.6749, F1=0.6770
  Val:   Acc=0.6968, F1=0.6961
  ✓ Best model saved (F1: 0.6961)


Epoch 5/5: 100%|███████████████| 1907/1907 [05:23<00:00,  5.89it/s, loss=0.4388]



Epoch 5/5:
  Train: Loss=1.1226, Acc=0.7247, F1=0.7230
  Val:   Acc=0.7115, F1=0.7096
  ✓ Best model saved (F1: 0.7096)

Loading best model (F1: 0.7096)


Testing: 100%|████████████████████████████████| 409/409 [00:24<00:00, 16.46it/s]



Full+FrozenEmbed Results:
  Accuracy:  0.7237
  Precision: 0.7215
  Recall:    0.7237
  F1 Score:  0.7169

ABLATION STUDY SUMMARY
   Configuration  Accuracy  Precision   Recall  F1-Score
  Full(Proposed)  0.735941   0.751805 0.735941  0.738207
       Full-ICFG  0.740831   0.747230 0.740831  0.741384
        Full-DFG  0.713936   0.738973 0.713936  0.714191
        Full-CDG  0.723716   0.761251 0.723716  0.733797
        ICFG+DFG  0.733496   0.744462 0.733496  0.721988
        ICFG+CDG  0.743276   0.742555 0.743276  0.738820
         DFG+CDG  0.748166   0.754336 0.748166  0.749387
        Full+AST  0.713936   0.729531 0.713936  0.717922
        Full+CFG  0.738386   0.746591 0.738386  0.739367
       Full+1GCN  0.726161   0.727238 0.726161  0.724525
       Full+3GCN  0.721271   0.731631 0.721271  0.715231
   Full-CodeText  0.195599   0.038259 0.195599  0.064000
Full+FrozenEmbed  0.723716   0.721523 0.723716  0.716934

Results saved to: ablation_results.csv


# traditonal classifers on devign dataset (codegraphnet embed)

In [ ]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, AdamW
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error
)
import xgboost as xgb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
import warnings
warnings.filterwarnings('ignore')


def load_and_sample_data(train_path, test_path):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    
    X_train = train_df.drop('label', axis=1).values
    y_train = train_df['label'].values
    X_test = test_df.drop('label', axis=1).values
    y_test = test_df['label'].values
    
    return X_train, y_train, X_test, y_test


def calculate_metrics(y_true, y_pred, y_pred_proba=None):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='binary', zero_division=0)
    recall = recall_score(y_true, y_pred, average='binary', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='binary', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    auc = 0.0
    if y_pred_proba is not None:
        try:
            if len(y_pred_proba.shape) > 1 and y_pred_proba.shape[1] == 2:
                auc = roc_auc_score(y_true, y_pred_proba[:, 1])
            else:
                auc = roc_auc_score(y_true, y_pred_proba)
        except:
            auc = 0.0
    
    return {
        'AUC': auc,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'MCC': mcc,
        'Kappa': kappa,
        'MSE': mse,
        'MAE': mae,
        'TP': tp,
        'FN': fn,
        'TN': tn,
        'FP': fp
    }


def print_results(model_name, metrics):
    print(f'\n{"="*80}')
    print(f'{model_name:^80}')
    print(f'{"="*80}')
    print(f'  AUC:       {metrics["AUC"]:.4f}')
    print(f'  Accuracy:  {metrics["Accuracy"]:.4f}')
    print(f'  Precision: {metrics["Precision"]:.4f}')
    print(f'  Recall:    {metrics["Recall"]:.4f}')
    print(f'  F1 Score:  {metrics["F1"]:.4f}')
    print(f'  MCC:       {metrics["MCC"]:.4f}')
    print(f'  Kappa:     {metrics["Kappa"]:.4f}')
    print(f'  MSE:       {metrics["MSE"]:.4f}')
    print(f'  MAE:       {metrics["MAE"]:.4f}')
    print(f'  TP:        {metrics["TP"]}')
    print(f'  FN:        {metrics["FN"]}')
    print(f'  TN:        {metrics["TN"]}')
    print(f'  FP:        {metrics["FP"]}')
    print(f'{"="*80}\n')


def train_gru(X_train, y_train, X_test, y_test):
    print('Training GRU...')
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    X_train_gru = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
    X_test_gru = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))
    
    model = Sequential()
    model.add(GRU(128, input_shape=(X_train_gru.shape[1], X_train_gru.shape[2]), return_sequences=True))
    model.add(Dropout(0.5))
    model.add(GRU(64))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    model.fit(X_train_gru, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=0)
    
    y_pred_proba = model.predict(X_test_gru, verbose=0).flatten()
    y_pred = (y_pred_proba > 0.5).astype(int)
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_results('GRU MODEL', metrics)
    return metrics


def train_xgboost(X_train, y_train, X_test, y_test):
    print('Training XGBoost...')
    model = xgb.XGBClassifier(eval_metric='logloss', random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_results('XGBOOST MODEL', metrics)
    return metrics


def train_svm(X_train, y_train, X_test, y_test):
    print('Training SVM...')
    model = SVC(kernel='linear', probability=True, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_results('SVM MODEL', metrics)
    return metrics


def train_deeptree(X_train, y_train, X_test, y_test):
    print('Training DeepTree...')
    dt_classifier = DecisionTreeClassifier(random_state=42, max_depth=30)
    dt_classifier.fit(X_train, y_train)
    
    X_train_transformed = dt_classifier.predict_proba(X_train)
    X_test_transformed = dt_classifier.predict_proba(X_test)
    
    model = Sequential([
        Dense(256, activation='relu', input_shape=(X_train_transformed.shape[1],)),
        Dense(128, activation='relu'),
        Dense(64, activation='relu'),
        Dense(32, activation='relu'),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(X_train_transformed, y_train, epochs=5, batch_size=16, validation_split=0.2, verbose=0)
    
    y_pred_proba = model.predict(X_test_transformed, verbose=0).flatten()
    y_pred = (y_pred_proba > 0.5).astype(int)
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_results('DEEPTREE MODEL', metrics)
    return metrics


def train_sgd(X_train, y_train, X_test, y_test):
    print('Training SGD...')
    model = SGDClassifier(loss='hinge', penalty='l2', alpha=1000, learning_rate='constant',
                         eta0=0.0001, max_iter=65, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    metrics = calculate_metrics(y_test, y_pred, None)
    print_results('SGD CLASSIFIER', metrics)
    return metrics


def train_decision_tree(X_train, y_train, X_test, y_test):
    print('Training Decision Tree...')
    model = DecisionTreeClassifier(max_depth=30, min_samples_split=5, min_samples_leaf=2, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_results('DECISION TREE', metrics)
    return metrics


def train_random_forest(X_train, y_train, X_test, y_test):
    print('Training Random Forest...')
    model = RandomForestClassifier(n_estimators=100, min_samples_split=2, max_depth=None,
                                  max_features='sqrt', random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_results('RANDOM FOREST', metrics)
    return metrics


def train_lstm(X_train, y_train, X_test, y_test):
    print('Training LSTM...')
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = Sequential([
        Dense(256, activation='relu', input_shape=(X_train_scaled.shape[1],)),
        Dropout(0.5),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                 loss='binary_crossentropy', metrics=['accuracy'])
    
    early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    model.fit(X_train_scaled, y_train, epochs=50, batch_size=32,
             validation_split=0.2, callbacks=[early_stopping], verbose=0)
    
    y_pred_proba = model.predict(X_test_scaled, verbose=0).flatten()
    y_pred = (y_pred_proba > 0.5).astype(int)
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_results('LSTM MODEL', metrics)
    return metrics


class NumericalDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return {'features': self.features[idx], 'labels': self.labels[idx]}


class NumericalTransformerClassifier(nn.Module):
    def __init__(self, input_dim, num_labels):
        super(NumericalTransformerClassifier, self).__init__()
        self.transformer = BertModel.from_pretrained('bert-base-uncased')
        self.embedding = nn.Linear(input_dim, self.transformer.config.hidden_size)
        self.dropout = nn.Dropout(0.5)
        self.classifier = nn.Linear(self.transformer.config.hidden_size, num_labels)

    def forward(self, features):
        embeddings = self.embedding(features)
        transformer_output = self.transformer(inputs_embeds=embeddings.unsqueeze(1)).last_hidden_state[:, 0, :]
        output = self.classifier(self.dropout(transformer_output))
        return output


def train_bert(X_train, y_train, X_test, y_test):
    print('Training BERT...')
    train_dataset = NumericalDataset(X_train, y_train)
    test_dataset = NumericalDataset(X_test, y_test)
    
    batch_size = 64
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    input_dim = X_train.shape[1]
    model = NumericalTransformerClassifier(input_dim, 2)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=2e-6, weight_decay=0.001)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    epochs = 40
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            features = batch['features'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    model.eval()
    predictions, actuals, all_outputs_prob = [], [], []
    with torch.no_grad():
        for batch in test_loader:
            features = batch['features'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(features)
            
            outputs_prob = nn.functional.softmax(outputs, dim=1).cpu().numpy()
            all_outputs_prob.extend(outputs_prob[:, 1])
            
            _, preds = torch.max(outputs, dim=1)
            predictions.extend(preds.cpu().numpy())
            actuals.extend(labels.cpu().numpy())
    
    y_pred = np.array(predictions)
    y_true = np.array(actuals)
    y_pred_proba = np.array(all_outputs_prob)
    
    metrics = calculate_metrics(y_true, y_pred, y_pred_proba)
    print_results('BERT TRANSFORMER', metrics)
    return metrics


if __name__ == '__main__':
    train_path = '/Users/akter/fahim/codegraph/traindev.csv'
    test_path = '/Users/akter/fahim/codegraph/testdev.csv'
    
    
    print('Loading and sampling data...')
    X_train, y_train, X_test, y_test = load_and_sample_data(train_path, test_path)
    
    num_classes = len(np.unique(y_train))
    
    print(f'\nDataset Information:')
    print(f'  Training samples:   {X_train.shape[0]}')
    print(f'  Test samples:       {X_test.shape[0]}')
    print(f'  Feature dimensions: {X_train.shape[1]}')
    print(f'  Number of classes:  {num_classes}')
    
    print(f'\nTraining Label Distribution:')
    unique, counts = np.unique(y_train, return_counts=True)
    for label, count in zip(unique, counts):
        print(f'  Class {label}: {count}')
    
    print(f'\nTest Label Distribution:')
    unique, counts = np.unique(y_test, return_counts=True)
    for label, count in zip(unique, counts):
        print(f'  Class {label}: {count}')
    
    results = {}
    
    results['GRU'] = train_gru(X_train, y_train, X_test, y_test)
    results['XGBoost'] = train_xgboost(X_train, y_train, X_test, y_test)
    results['SVM'] = train_svm(X_train, y_train, X_test, y_test)
    results['DeepTree'] = train_deeptree(X_train, y_train, X_test, y_test)
    results['SGD'] = train_sgd(X_train, y_train, X_test, y_test)
    results['Decision Tree'] = train_decision_tree(X_train, y_train, X_test, y_test)
    results['Random Forest'] = train_random_forest(X_train, y_train, X_test, y_test)
    results['LSTM'] = train_lstm(X_train, y_train, X_test, y_test)
    results['BERT'] = train_bert(X_train, y_train, X_test, y_test)
    
    print('\n' + '='*130)
    print(f'{"SUMMARY OF ALL MODELS":^130}')
    print('='*130)
    print(f'{"Model":<15} {"AUC":<8} {"Acc":<8} {"Pre":<8} {"Rec":<8} {"F1":<8} {"MCC":<8} {"Kappa":<8} {"MSE":<8} {"MAE":<8} {"TP":<6} {"FN":<6} {"TN":<6} {"FP":<6}')
    print('-'*130)
    
    for model_name, metrics in results.items():
        print(f'{model_name:<15} {metrics["AUC"]:<8.4f} {metrics["Accuracy"]:<8.4f} {metrics["Precision"]:<8.4f} '
              f'{metrics["Recall"]:<8.4f} {metrics["F1"]:<8.4f} {metrics["MCC"]:<8.4f} {metrics["Kappa"]:<8.4f} '
              f'{metrics["MSE"]:<8.4f} {metrics["MAE"]:<8.4f} {metrics["TP"]:<6} {metrics["FN"]:<6} {metrics["TN"]:<6} {metrics["FP"]:<6}')
    
    print('='*130)
    
    best_model = max(results.items(), key=lambda x: x[1]['F1'])
    print(f'\nBest Model: {best_model[0]} (F1 Score: {best_model[1]["F1"]:.4f})')

# transfoermer style classifer on devign daatset(codegraphnet embed)

In [ ]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, mean_squared_error, mean_absolute_error
)
from pytorch_tabnet.tab_model import TabNetClassifier
import warnings
warnings.filterwarnings('ignore')


def load_and_sample_data(train_path, test_path):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    
    X_train = train_df.drop('label', axis=1).values
    y_train = train_df['label'].values
    X_test = test_df.drop('label', axis=1).values
    y_test = test_df['label'].values
    
    return X_train, y_train, X_test, y_test


def calculate_metrics(y_true, y_pred, y_pred_proba=None):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='binary', zero_division=0)
    recall = recall_score(y_true, y_pred, average='binary', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='binary', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    auc = 0.0
    if y_pred_proba is not None:
        try:
            if y_pred_proba.ndim == 2:
                auc = roc_auc_score(y_true, y_pred_proba[:, 1])
            else:
                auc = roc_auc_score(y_true, y_pred_proba)
        except:
            auc = 0.0
    
    return {
        'AUC': auc, 'Accuracy': accuracy, 'Precision': precision,
        'Recall': recall, 'F1': f1, 'MCC': mcc, 'Kappa': kappa,
        'MSE': mse, 'MAE': mae, 'TP': tp, 'FN': fn
    }


def print_results(model_name, metrics):
    print(f'\n{"="*80}')
    print(f'{model_name:^80}')
    print(f'{"="*80}')
    print(f'  AUC:       {metrics["AUC"]:.4f}')
    print(f'  Accuracy:  {metrics["Accuracy"]:.4f}')
    print(f'  Precision: {metrics["Precision"]:.4f}')
    print(f'  Recall:    {metrics["Recall"]:.4f}')
    print(f'  F1 Score:  {metrics["F1"]:.4f}')
    print(f'  MCC:       {metrics["MCC"]:.4f}')
    print(f'  Kappa:     {metrics["Kappa"]:.4f}')
    print(f'  MSE:       {metrics["MSE"]:.4f}')
    print(f'  MAE:       {metrics["MAE"]:.4f}')
    print(f'  TP:        {metrics["TP"]}')
    print(f'  FN:        {metrics["FN"]}')
    print(f'{"="*80}\n')


class AutoIntModel(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dim=256, num_heads=8, num_layers=3, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, hidden_dim)
        
        self.attention_layers = nn.ModuleList([
            nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
            for _ in range(num_layers)
        ])
        
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        x = self.input_projection(x).unsqueeze(1)
        
        for attn, norm in zip(self.attention_layers, self.norms):
            attn_out, _ = attn(x, x, x)
            x = norm(x + self.dropout(attn_out))
        
        x = x.squeeze(1)
        return self.classifier(x)


class FTTransformerModel(nn.Module):
    def __init__(self, input_dim, num_classes, d_model=256, num_heads=8, num_layers=6, dim_ff=512, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, dim_feedforward=dim_ff,
            dropout=dropout, activation='relu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        batch_size = x.size(0)
        x = self.input_projection(x).unsqueeze(1)
        
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        x = self.transformer(x)
        cls_output = x[:, 0]
        
        return self.classifier(cls_output)


class TabTransformerModel(nn.Module):
    def __init__(self, input_dim, num_classes, d_model=256, num_heads=8, num_layers=6, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, dim_feedforward=4*d_model,
            dropout=dropout, activation='relu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        x = self.input_projection(x).unsqueeze(1)
        x = self.transformer(x)
        x = x.squeeze(1)
        return self.classifier(x)


class SAINTModel(nn.Module):
    def __init__(self, input_dim, num_classes, d_model=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, d_model)
        
        self.attention_layers = nn.ModuleList([
            nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
            for _ in range(num_layers)
        ])
        
        self.ffn_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, 4*d_model),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(4*d_model, d_model),
                nn.Dropout(dropout)
            ) for _ in range(num_layers)
        ])
        
        self.norms1 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(num_layers)])
        self.norms2 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(num_layers)])
        
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        x = self.input_projection(x).unsqueeze(1)
        
        for attn, ffn, norm1, norm2 in zip(self.attention_layers, self.ffn_layers, self.norms1, self.norms2):
            attn_out, _ = attn(x, x, x)
            x = norm1(x + attn_out)
            
            ffn_out = ffn(x)
            x = norm2(x + ffn_out)
        
        x = x.squeeze(1)
        return self.classifier(x)


class TabularPerceptron(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dims=[512, 256, 128], dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, num_classes))
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)


class TensorDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
        
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


def train_pytorch_model(model, train_loader, test_loader, device, epochs=50, lr=0.001):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for features, labels in test_loader:
            features = features.to(device)
            outputs = model(features)
            probs = F.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


def train_autoint(X_train, y_train, X_test, y_test, num_classes, device):
    print('Training AutoInt...')
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    train_dataset = TensorDataset(X_train_scaled, y_train)
    test_dataset = TensorDataset(X_test_scaled, y_test)
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    
    model = AutoIntModel(X_train.shape[1], num_classes, hidden_dim=256, num_heads=8, num_layers=3)
    y_true, y_pred, y_pred_proba = train_pytorch_model(model, train_loader, test_loader, device, epochs=50, lr=0.001)
    
    metrics = calculate_metrics(y_true, y_pred, y_pred_proba)
    print_results('AUTOINT', metrics)
    return metrics


def train_tabnet(X_train, y_train, X_test, y_test, num_classes):
    print('Training TabNet...')
    model = TabNetClassifier(
        n_d=64, n_a=64, n_steps=5, gamma=1.5,
        n_independent=2, n_shared=2,
        lambda_sparse=1e-4, momentum=0.3,
        clip_value=2.0, optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2),
        scheduler_params={"step_size":50, "gamma":0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type='entmax', verbose=0
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        max_epochs=100, patience=20,
        batch_size=256, virtual_batch_size=128
    )
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_results('TABNET', metrics)
    return metrics


def train_fttransformer(X_train, y_train, X_test, y_test, num_classes, device):
    print('Training FT-Transformer...')
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    train_dataset = TensorDataset(X_train_scaled, y_train)
    test_dataset = TensorDataset(X_test_scaled, y_test)
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    
    model = FTTransformerModel(X_train.shape[1], num_classes, d_model=256, num_heads=8, num_layers=6)
    y_true, y_pred, y_pred_proba = train_pytorch_model(model, train_loader, test_loader, device, epochs=50, lr=0.0001)
    
    metrics = calculate_metrics(y_true, y_pred, y_pred_proba)
    print_results('FT-TRANSFORMER', metrics)
    return metrics


def train_tabtransformer(X_train, y_train, X_test, y_test, num_classes, device):
    print('Training TabTransformer...')
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    train_dataset = TensorDataset(X_train_scaled, y_train)
    test_dataset = TensorDataset(X_test_scaled, y_test)
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    
    model = TabTransformerModel(X_train.shape[1], num_classes, d_model=256, num_heads=8, num_layers=6)
    y_true, y_pred, y_pred_proba = train_pytorch_model(model, train_loader, test_loader, device, epochs=50, lr=0.001)
    
    metrics = calculate_metrics(y_true, y_pred, y_pred_proba)
    print_results('TABTRANSFORMER', metrics)
    return metrics


def train_saint(X_train, y_train, X_test, y_test, num_classes, device):
    print('Training SAINT...')
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    train_dataset = TensorDataset(X_train_scaled, y_train)
    test_dataset = TensorDataset(X_test_scaled, y_test)
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    
    model = SAINTModel(X_train.shape[1], num_classes, d_model=256, num_heads=8, num_layers=4)
    y_true, y_pred, y_pred_proba = train_pytorch_model(model, train_loader, test_loader, device, epochs=50, lr=0.001)
    
    metrics = calculate_metrics(y_true, y_pred, y_pred_proba)
    print_results('SAINT', metrics)
    return metrics


def train_tabular_perceptron(X_train, y_train, X_test, y_test, num_classes, device):
    print('Training Tabular Perceptron...')
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    train_dataset = TensorDataset(X_train_scaled, y_train)
    test_dataset = TensorDataset(X_test_scaled, y_test)
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    
    model = TabularPerceptron(X_train.shape[1], num_classes, hidden_dims=[512, 256, 128])
    y_true, y_pred, y_pred_proba = train_pytorch_model(model, train_loader, test_loader, device, epochs=50, lr=0.001)
    
    metrics = calculate_metrics(y_true, y_pred, y_pred_proba)
    print_results('TABULAR PERCEPTRON', metrics)
    return metrics


if __name__ == '__main__':
    train_path = '/Users/akter/fahim/codegraph/traindev.csv'
    test_path = '/Users/akter/fahim/codegraph/testdev.csv'
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}\n')
    
    print('Loading and sampling data...')
    X_train, y_train, X_test, y_test = load_and_sample_data(train_path, test_path)
    
    num_classes = len(np.unique(y_train))
    
    print(f'\nDataset Information:')
    print(f'  Training samples:   {X_train.shape[0]}')
    print(f'  Test samples:       {X_test.shape[0]}')
    print(f'  Feature dimensions: {X_train.shape[1]}')
    print(f'  Number of classes:  {num_classes}')
    
    print(f'\nTraining Label Distribution:')
    unique, counts = np.unique(y_train, return_counts=True)
    for label, count in zip(unique, counts):
        print(f'  Class {label}: {count}')
    
    print(f'\nTest Label Distribution:')
    unique, counts = np.unique(y_test, return_counts=True)
    for label, count in zip(unique, counts):
        print(f'  Class {label}: {count}')
    
    results = {}
    
    results['AutoInt'] = train_autoint(X_train, y_train, X_test, y_test, num_classes, device)
    results['TabNet'] = train_tabnet(X_train, y_train, X_test, y_test, num_classes)
    results['FT-Transformer'] = train_fttransformer(X_train, y_train, X_test, y_test, num_classes, device)
    results['TabTransformer'] = train_tabtransformer(X_train, y_train, X_test, y_test, num_classes, device)
    results['SAINT'] = train_saint(X_train, y_train, X_test, y_test, num_classes, device)
    results['Tabular Perceptron'] = train_tabular_perceptron(X_train, y_train, X_test, y_test, num_classes, device)
    
    print('\n' + '='*120)
    print(f'{"SUMMARY OF ALL TRANSFORMER MODELS":^120}')
    print('='*120)
    print(f'{"Model":<20} {"AUC":<8} {"Acc":<8} {"Pre":<8} {"Rec":<8} {"F1":<8} {"MCC":<8} {"Kappa":<8} {"MSE":<8} {"MAE":<8} {"TP":<6} {"FN":<6}')
    print('-'*120)
    
    for model_name, metrics in results.items():
        print(f'{model_name:<20} {metrics["AUC"]:<8.4f} {metrics["Accuracy"]:<8.4f} {metrics["Precision"]:<8.4f} '
              f'{metrics["Recall"]:<8.4f} {metrics["F1"]:<8.4f} {metrics["MCC"]:<8.4f} {metrics["Kappa"]:<8.4f} '
              f'{metrics["MSE"]:<8.4f} {metrics["MAE"]:<8.4f} {metrics["TP"]:<6} {metrics["FN"]:<6}')
    
    print('='*120)
    
    best_model = max(results.items(), key=lambda x: x[1]['F1'])
    print(f'\nBest Model: {best_model[0]} (F1 Score: {best_model[1]["F1"]:.4f})')

# codexglue dataset traditional classifer (codegrpahnet embed)

In [ ]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, AdamW
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error
)
import xgboost as xgb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
import warnings
warnings.filterwarnings('ignore')


def load_and_sample_data(train_path, test_path):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    

    X_train = train_df.drop('label', axis=1).values
    y_train = train_df['label'].values
    X_test = test_df.drop('label', axis=1).values
    y_test = test_df['label'].values
    
    return X_train, y_train, X_test, y_test


def calculate_metrics(y_true, y_pred, y_pred_proba=None):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='binary', zero_division=0)
    recall = recall_score(y_true, y_pred, average='binary', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='binary', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    auc = 0.0
    if y_pred_proba is not None:
        try:
            if len(y_pred_proba.shape) > 1 and y_pred_proba.shape[1] == 2:
                auc = roc_auc_score(y_true, y_pred_proba[:, 1])
            else:
                auc = roc_auc_score(y_true, y_pred_proba)
        except:
            auc = 0.0
    
    return {
        'AUC': auc,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'MCC': mcc,
        'Kappa': kappa,
        'MSE': mse,
        'MAE': mae,
        'TP': tp,
        'FN': fn,
        'TN': tn,
        'FP': fp
    }


def print_results(model_name, metrics):
    print(f'\n{"="*80}')
    print(f'{model_name:^80}')
    print(f'{"="*80}')
    print(f'  AUC:       {metrics["AUC"]:.4f}')
    print(f'  Accuracy:  {metrics["Accuracy"]:.4f}')
    print(f'  Precision: {metrics["Precision"]:.4f}')
    print(f'  Recall:    {metrics["Recall"]:.4f}')
    print(f'  F1 Score:  {metrics["F1"]:.4f}')
    print(f'  MCC:       {metrics["MCC"]:.4f}')
    print(f'  Kappa:     {metrics["Kappa"]:.4f}')
    print(f'  MSE:       {metrics["MSE"]:.4f}')
    print(f'  MAE:       {metrics["MAE"]:.4f}')
    print(f'  TP:        {metrics["TP"]}')
    print(f'  FN:        {metrics["FN"]}')
    print(f'  TN:        {metrics["TN"]}')
    print(f'  FP:        {metrics["FP"]}')
    print(f'{"="*80}\n')


def train_gru(X_train, y_train, X_test, y_test):
    print('Training GRU...')
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    X_train_gru = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
    X_test_gru = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))
    
    model = Sequential()
    model.add(GRU(128, input_shape=(X_train_gru.shape[1], X_train_gru.shape[2]), return_sequences=True))
    model.add(Dropout(0.5))
    model.add(GRU(64))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    model.fit(X_train_gru, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=0)
    
    y_pred_proba = model.predict(X_test_gru, verbose=0).flatten()
    y_pred = (y_pred_proba > 0.5).astype(int)
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_results('GRU MODEL', metrics)
    return metrics


def train_xgboost(X_train, y_train, X_test, y_test):
    print('Training XGBoost...')
    model = xgb.XGBClassifier(eval_metric='logloss', random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_results('XGBOOST MODEL', metrics)
    return metrics


def train_svm(X_train, y_train, X_test, y_test):
    print('Training SVM...')
    model = SVC(kernel='linear', probability=True, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_results('SVM MODEL', metrics)
    return metrics


def train_deeptree(X_train, y_train, X_test, y_test):
    print('Training DeepTree...')
    dt_classifier = DecisionTreeClassifier(random_state=42, max_depth=30)
    dt_classifier.fit(X_train, y_train)
    
    X_train_transformed = dt_classifier.predict_proba(X_train)
    X_test_transformed = dt_classifier.predict_proba(X_test)
    
    model = Sequential([
        Dense(256, activation='relu', input_shape=(X_train_transformed.shape[1],)),
        Dense(128, activation='relu'),
        Dense(64, activation='relu'),
        Dense(32, activation='relu'),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(X_train_transformed, y_train, epochs=5, batch_size=16, validation_split=0.2, verbose=0)
    
    y_pred_proba = model.predict(X_test_transformed, verbose=0).flatten()
    y_pred = (y_pred_proba > 0.5).astype(int)
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_results('DEEPTREE MODEL', metrics)
    return metrics


def train_sgd(X_train, y_train, X_test, y_test):
    print('Training SGD...')
    model = SGDClassifier(loss='hinge', penalty='l2', alpha=1000, learning_rate='constant',
                         eta0=0.0001, max_iter=65, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    metrics = calculate_metrics(y_test, y_pred, None)
    print_results('SGD CLASSIFIER', metrics)
    return metrics


def train_decision_tree(X_train, y_train, X_test, y_test):
    print('Training Decision Tree...')
    model = DecisionTreeClassifier(max_depth=30, min_samples_split=5, min_samples_leaf=2, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_results('DECISION TREE', metrics)
    return metrics


def train_random_forest(X_train, y_train, X_test, y_test):
    print('Training Random Forest...')
    model = RandomForestClassifier(n_estimators=100, min_samples_split=2, max_depth=None,
                                  max_features='sqrt', random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_results('RANDOM FOREST', metrics)
    return metrics


def train_lstm(X_train, y_train, X_test, y_test):
    print('Training LSTM...')
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = Sequential([
        Dense(256, activation='relu', input_shape=(X_train_scaled.shape[1],)),
        Dropout(0.5),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                 loss='binary_crossentropy', metrics=['accuracy'])
    
    early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    model.fit(X_train_scaled, y_train, epochs=50, batch_size=32,
             validation_split=0.2, callbacks=[early_stopping], verbose=0)
    
    y_pred_proba = model.predict(X_test_scaled, verbose=0).flatten()
    y_pred = (y_pred_proba > 0.5).astype(int)
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_results('LSTM MODEL', metrics)
    return metrics


class NumericalDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return {'features': self.features[idx], 'labels': self.labels[idx]}


class NumericalTransformerClassifier(nn.Module):
    def __init__(self, input_dim, num_labels):
        super(NumericalTransformerClassifier, self).__init__()
        self.transformer = BertModel.from_pretrained('bert-base-uncased')
        self.embedding = nn.Linear(input_dim, self.transformer.config.hidden_size)
        self.dropout = nn.Dropout(0.5)
        self.classifier = nn.Linear(self.transformer.config.hidden_size, num_labels)

    def forward(self, features):
        embeddings = self.embedding(features)
        transformer_output = self.transformer(inputs_embeds=embeddings.unsqueeze(1)).last_hidden_state[:, 0, :]
        output = self.classifier(self.dropout(transformer_output))
        return output


def train_bert(X_train, y_train, X_test, y_test):
    print('Training BERT...')
    train_dataset = NumericalDataset(X_train, y_train)
    test_dataset = NumericalDataset(X_test, y_test)
    
    batch_size = 64
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    input_dim = X_train.shape[1]
    model = NumericalTransformerClassifier(input_dim, 2)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=2e-6, weight_decay=0.001)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    epochs = 40
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            features = batch['features'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    model.eval()
    predictions, actuals, all_outputs_prob = [], [], []
    with torch.no_grad():
        for batch in test_loader:
            features = batch['features'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(features)
            
            outputs_prob = nn.functional.softmax(outputs, dim=1).cpu().numpy()
            all_outputs_prob.extend(outputs_prob[:, 1])
            
            _, preds = torch.max(outputs, dim=1)
            predictions.extend(preds.cpu().numpy())
            actuals.extend(labels.cpu().numpy())
    
    y_pred = np.array(predictions)
    y_true = np.array(actuals)
    y_pred_proba = np.array(all_outputs_prob)
    
    metrics = calculate_metrics(y_true, y_pred, y_pred_proba)
    print_results('BERT TRANSFORMER', metrics)
    return metrics


if __name__ == '__main__':
    train_path = '/Users/akter/fahim/codegraph/traincodex.csv'
    test_path = '/Users/akter/fahim/codegraph/testcodex.csv'
    
    print('Loading and sampling data...')
    X_train, y_train, X_test, y_test = load_and_sample_data(train_path, test_path)
    
    num_classes = len(np.unique(y_train))
    
    print(f'\nDataset Information:')
    print(f'  Training samples:   {X_train.shape[0]}')
    print(f'  Test samples:       {X_test.shape[0]}')
    print(f'  Feature dimensions: {X_train.shape[1]}')
    print(f'  Number of classes:  {num_classes}')
    
    print(f'\nTraining Label Distribution:')
    unique, counts = np.unique(y_train, return_counts=True)
    for label, count in zip(unique, counts):
        print(f'  Class {label}: {count}')
    
    print(f'\nTest Label Distribution:')
    unique, counts = np.unique(y_test, return_counts=True)
    for label, count in zip(unique, counts):
        print(f'  Class {label}: {count}')
    
    results = {}
    
    results['GRU'] = train_gru(X_train, y_train, X_test, y_test)
    results['XGBoost'] = train_xgboost(X_train, y_train, X_test, y_test)
    results['SVM'] = train_svm(X_train, y_train, X_test, y_test)
    results['DeepTree'] = train_deeptree(X_train, y_train, X_test, y_test)
    results['SGD'] = train_sgd(X_train, y_train, X_test, y_test)
    results['Decision Tree'] = train_decision_tree(X_train, y_train, X_test, y_test)
    results['Random Forest'] = train_random_forest(X_train, y_train, X_test, y_test)
    results['LSTM'] = train_lstm(X_train, y_train, X_test, y_test)
    results['BERT'] = train_bert(X_train, y_train, X_test, y_test)
    
    print('\n' + '='*130)
    print(f'{"SUMMARY OF ALL MODELS":^130}')
    print('='*130)
    print(f'{"Model":<15} {"AUC":<8} {"Acc":<8} {"Pre":<8} {"Rec":<8} {"F1":<8} {"MCC":<8} {"Kappa":<8} {"MSE":<8} {"MAE":<8} {"TP":<6} {"FN":<6} {"TN":<6} {"FP":<6}')
    print('-'*130)
    
    for model_name, metrics in results.items():
        print(f'{model_name:<15} {metrics["AUC"]:<8.4f} {metrics["Accuracy"]:<8.4f} {metrics["Precision"]:<8.4f} '
              f'{metrics["Recall"]:<8.4f} {metrics["F1"]:<8.4f} {metrics["MCC"]:<8.4f} {metrics["Kappa"]:<8.4f} '
              f'{metrics["MSE"]:<8.4f} {metrics["MAE"]:<8.4f} {metrics["TP"]:<6} {metrics["FN"]:<6} {metrics["TN"]:<6} {metrics["FP"]:<6}')
    
    print('='*130)
    
    best_model = max(results.items(), key=lambda x: x[1]['F1'])
    print(f'\nBest Model: {best_model[0]} (F1 Score: {best_model[1]["F1"]:.4f})')

# trasnformer style classifers cxodexglue (codegraphnet embed)

In [ ]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, mean_squared_error, mean_absolute_error
)
from pytorch_tabnet.tab_model import TabNetClassifier
import warnings
warnings.filterwarnings('ignore')


def load_and_sample_data(train_path, test_path):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    
    X_train = train_df.drop('label', axis=1).values
    y_train = train_df['label'].values
    X_test = test_df.drop('label', axis=1).values
    y_test = test_df['label'].values
    
    return X_train, y_train, X_test, y_test


def calculate_metrics(y_true, y_pred, y_pred_proba=None):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='binary', zero_division=0)
    recall = recall_score(y_true, y_pred, average='binary', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='binary', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    auc = 0.0
    if y_pred_proba is not None:
        try:
            if y_pred_proba.ndim == 2:
                auc = roc_auc_score(y_true, y_pred_proba[:, 1])
            else:
                auc = roc_auc_score(y_true, y_pred_proba)
        except:
            auc = 0.0
    
    return {
        'AUC': auc, 'Accuracy': accuracy, 'Precision': precision,
        'Recall': recall, 'F1': f1, 'MCC': mcc, 'Kappa': kappa,
        'MSE': mse, 'MAE': mae, 'TP': tp, 'FN': fn
    }


def print_results(model_name, metrics):
    print(f'\n{"="*80}')
    print(f'{model_name:^80}')
    print(f'{"="*80}')
    print(f'  AUC:       {metrics["AUC"]:.4f}')
    print(f'  Accuracy:  {metrics["Accuracy"]:.4f}')
    print(f'  Precision: {metrics["Precision"]:.4f}')
    print(f'  Recall:    {metrics["Recall"]:.4f}')
    print(f'  F1 Score:  {metrics["F1"]:.4f}')
    print(f'  MCC:       {metrics["MCC"]:.4f}')
    print(f'  Kappa:     {metrics["Kappa"]:.4f}')
    print(f'  MSE:       {metrics["MSE"]:.4f}')
    print(f'  MAE:       {metrics["MAE"]:.4f}')
    print(f'  TP:        {metrics["TP"]}')
    print(f'  FN:        {metrics["FN"]}')
    print(f'{"="*80}\n')


class AutoIntModel(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dim=256, num_heads=8, num_layers=3, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, hidden_dim)
        
        self.attention_layers = nn.ModuleList([
            nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
            for _ in range(num_layers)
        ])
        
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        x = self.input_projection(x).unsqueeze(1)
        
        for attn, norm in zip(self.attention_layers, self.norms):
            attn_out, _ = attn(x, x, x)
            x = norm(x + self.dropout(attn_out))
        
        x = x.squeeze(1)
        return self.classifier(x)


class FTTransformerModel(nn.Module):
    def __init__(self, input_dim, num_classes, d_model=256, num_heads=8, num_layers=6, dim_ff=512, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, dim_feedforward=dim_ff,
            dropout=dropout, activation='relu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        batch_size = x.size(0)
        x = self.input_projection(x).unsqueeze(1)
        
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        x = self.transformer(x)
        cls_output = x[:, 0]
        
        return self.classifier(cls_output)


class TabTransformerModel(nn.Module):
    def __init__(self, input_dim, num_classes, d_model=256, num_heads=8, num_layers=6, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, dim_feedforward=4*d_model,
            dropout=dropout, activation='relu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        x = self.input_projection(x).unsqueeze(1)
        x = self.transformer(x)
        x = x.squeeze(1)
        return self.classifier(x)


class SAINTModel(nn.Module):
    def __init__(self, input_dim, num_classes, d_model=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, d_model)
        
        self.attention_layers = nn.ModuleList([
            nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
            for _ in range(num_layers)
        ])
        
        self.ffn_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, 4*d_model),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(4*d_model, d_model),
                nn.Dropout(dropout)
            ) for _ in range(num_layers)
        ])
        
        self.norms1 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(num_layers)])
        self.norms2 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(num_layers)])
        
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        x = self.input_projection(x).unsqueeze(1)
        
        for attn, ffn, norm1, norm2 in zip(self.attention_layers, self.ffn_layers, self.norms1, self.norms2):
            attn_out, _ = attn(x, x, x)
            x = norm1(x + attn_out)
            
            ffn_out = ffn(x)
            x = norm2(x + ffn_out)
        
        x = x.squeeze(1)
        return self.classifier(x)


class TabularPerceptron(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dims=[512, 256, 128], dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, num_classes))
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)


class TensorDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
        
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


def train_pytorch_model(model, train_loader, test_loader, device, epochs=50, lr=0.001):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for features, labels in test_loader:
            features = features.to(device)
            outputs = model(features)
            probs = F.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


def train_autoint(X_train, y_train, X_test, y_test, num_classes, device):
    print('Training AutoInt...')
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    train_dataset = TensorDataset(X_train_scaled, y_train)
    test_dataset = TensorDataset(X_test_scaled, y_test)
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    
    model = AutoIntModel(X_train.shape[1], num_classes, hidden_dim=256, num_heads=8, num_layers=3)
    y_true, y_pred, y_pred_proba = train_pytorch_model(model, train_loader, test_loader, device, epochs=50, lr=0.001)
    
    metrics = calculate_metrics(y_true, y_pred, y_pred_proba)
    print_results('AUTOINT', metrics)
    return metrics


def train_tabnet(X_train, y_train, X_test, y_test, num_classes):
    print('Training TabNet...')
    model = TabNetClassifier(
        n_d=64, n_a=64, n_steps=5, gamma=1.5,
        n_independent=2, n_shared=2,
        lambda_sparse=1e-4, momentum=0.3,
        clip_value=2.0, optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2),
        scheduler_params={"step_size":50, "gamma":0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type='entmax', verbose=0
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        max_epochs=100, patience=20,
        batch_size=256, virtual_batch_size=128
    )
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)
    
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba)
    print_results('TABNET', metrics)
    return metrics


def train_fttransformer(X_train, y_train, X_test, y_test, num_classes, device):
    print('Training FT-Transformer...')
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    train_dataset = TensorDataset(X_train_scaled, y_train)
    test_dataset = TensorDataset(X_test_scaled, y_test)
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    
    model = FTTransformerModel(X_train.shape[1], num_classes, d_model=256, num_heads=8, num_layers=6)
    y_true, y_pred, y_pred_proba = train_pytorch_model(model, train_loader, test_loader, device, epochs=50, lr=0.0001)
    
    metrics = calculate_metrics(y_true, y_pred, y_pred_proba)
    print_results('FT-TRANSFORMER', metrics)
    return metrics


def train_tabtransformer(X_train, y_train, X_test, y_test, num_classes, device):
    print('Training TabTransformer...')
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    train_dataset = TensorDataset(X_train_scaled, y_train)
    test_dataset = TensorDataset(X_test_scaled, y_test)
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    
    model = TabTransformerModel(X_train.shape[1], num_classes, d_model=256, num_heads=8, num_layers=6)
    y_true, y_pred, y_pred_proba = train_pytorch_model(model, train_loader, test_loader, device, epochs=50, lr=0.001)
    
    metrics = calculate_metrics(y_true, y_pred, y_pred_proba)
    print_results('TABTRANSFORMER', metrics)
    return metrics


def train_saint(X_train, y_train, X_test, y_test, num_classes, device):
    print('Training SAINT...')
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    train_dataset = TensorDataset(X_train_scaled, y_train)
    test_dataset = TensorDataset(X_test_scaled, y_test)
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    
    model = SAINTModel(X_train.shape[1], num_classes, d_model=256, num_heads=8, num_layers=4)
    y_true, y_pred, y_pred_proba = train_pytorch_model(model, train_loader, test_loader, device, epochs=50, lr=0.001)
    
    metrics = calculate_metrics(y_true, y_pred, y_pred_proba)
    print_results('SAINT', metrics)
    return metrics


def train_tabular_perceptron(X_train, y_train, X_test, y_test, num_classes, device):
    print('Training Tabular Perceptron...')
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    train_dataset = TensorDataset(X_train_scaled, y_train)
    test_dataset = TensorDataset(X_test_scaled, y_test)
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    
    model = TabularPerceptron(X_train.shape[1], num_classes, hidden_dims=[512, 256, 128])
    y_true, y_pred, y_pred_proba = train_pytorch_model(model, train_loader, test_loader, device, epochs=50, lr=0.001)
    
    metrics = calculate_metrics(y_true, y_pred, y_pred_proba)
    print_results('TABULAR PERCEPTRON', metrics)
    return metrics


if __name__ == '__main__':
    train_path = '/Users/akter/fahim/codegraph/traincodex.csv'
    test_path = '/Users/akter/fahim/codegraph/testcodex.csv'
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}\n')
    
    print('Loading and sampling data...')
    X_train, y_train, X_test, y_test = load_and_sample_data(train_path, test_path)
    
    num_classes = len(np.unique(y_train))
    
    print(f'\nDataset Information:')
    print(f'  Training samples:   {X_train.shape[0]}')
    print(f'  Test samples:       {X_test.shape[0]}')
    print(f'  Feature dimensions: {X_train.shape[1]}')
    print(f'  Number of classes:  {num_classes}')
    
    print(f'\nTraining Label Distribution:')
    unique, counts = np.unique(y_train, return_counts=True)
    for label, count in zip(unique, counts):
        print(f'  Class {label}: {count}')
    
    print(f'\nTest Label Distribution:')
    unique, counts = np.unique(y_test, return_counts=True)
    for label, count in zip(unique, counts):
        print(f'  Class {label}: {count}')
    
    results = {}
    
    results['AutoInt'] = train_autoint(X_train, y_train, X_test, y_test, num_classes, device)
    results['TabNet'] = train_tabnet(X_train, y_train, X_test, y_test, num_classes)
    results['FT-Transformer'] = train_fttransformer(X_train, y_train, X_test, y_test, num_classes, device)
    results['TabTransformer'] = train_tabtransformer(X_train, y_train, X_test, y_test, num_classes, device)
    results['SAINT'] = train_saint(X_train, y_train, X_test, y_test, num_classes, device)
    results['Tabular Perceptron'] = train_tabular_perceptron(X_train, y_train, X_test, y_test, num_classes, device)
    
    print('\n' + '='*120)
    print(f'{"SUMMARY OF ALL TRANSFORMER MODELS":^120}')
    print('='*120)
    print(f'{"Model":<20} {"AUC":<8} {"Acc":<8} {"Pre":<8} {"Rec":<8} {"F1":<8} {"MCC":<8} {"Kappa":<8} {"MSE":<8} {"MAE":<8} {"TP":<6} {"FN":<6}')
    print('-'*120)
    
    for model_name, metrics in results.items():
        print(f'{model_name:<20} {metrics["AUC"]:<8.4f} {metrics["Accuracy"]:<8.4f} {metrics["Precision"]:<8.4f} '
              f'{metrics["Recall"]:<8.4f} {metrics["F1"]:<8.4f} {metrics["MCC"]:<8.4f} {metrics["Kappa"]:<8.4f} '
              f'{metrics["MSE"]:<8.4f} {metrics["MAE"]:<8.4f} {metrics["TP"]:<6} {metrics["FN"]:<6}')
    
    print('='*120)
    
    best_model = max(results.items(), key=lambda x: x[1]['F1'])
    print(f'\nBest Model: {best_model[0]} (F1 Score: {best_model[1]["F1"]:.4f})')


# 10 fold evalution of codegraphnet embed dataset for traditional classifer

In [ ]:

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, 
                             matthews_corrcoef, cohen_kappa_score, 
                             mean_squared_error, mean_absolute_error, roc_auc_score)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import label_binarize
import warnings
warnings.filterwarnings('ignore')


def load_data(train_path, test_path):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    
    X_train = train_df.drop('label', axis=1).values
    y_train = train_df['label'].values
    X_test = test_df.drop('label', axis=1).values
    y_test = test_df['label'].values
    
    return X_train, y_train, X_test, y_test


def calculate_metrics(y_true, y_pred, y_pred_proba, classes):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    
    if len(classes) == 2:
        auc = roc_auc_score(y_true, y_pred_proba[:, 1])
    else:
        y_true_bin = label_binarize(y_true, classes=classes)
        auc = roc_auc_score(y_true_bin, y_pred_proba, average='weighted', multi_class='ovr')
    
    return {
        'AUC': auc,
        'Acc': acc,
        'Pre': precision,
        'Rec': recall,
        'F1': f1,
        'MCC': mcc,
        'Kappa': kappa,
        'MSE': mse,
        'MAE': mae
    }


def get_classifier(model_name):
    if model_name == 'Random Forest':
        return RandomForestClassifier(n_estimators=200, max_depth=20, min_samples_split=5, 
                                     min_samples_leaf=2, random_state=42, n_jobs=-1)
    elif model_name == 'Gradient Boosting':
        return GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, 
                                         max_depth=5, random_state=42)
    elif model_name == 'XGBoost':
        return XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=6, 
                           random_state=42, eval_metric='mlogloss', use_label_encoder=False)
    elif model_name == 'SVM':
        return SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42, probability=True)
    elif model_name == 'Logistic Regression':
        return LogisticRegression(max_iter=1000, C=1.0, random_state=42, n_jobs=-1)
    elif model_name == 'MLP':
        return MLPClassifier(hidden_layer_sizes=(256, 128, 64), activation='relu', 
                           solver='adam', alpha=0.0001, learning_rate='adaptive', 
                           max_iter=500, random_state=42)


def cross_validate_on_train(X_train, y_train, model_name, n_splits=10, n_iterations=5):
    classes = np.unique(y_train)
    all_iteration_results = []
    
    for iteration in range(n_iterations):
        print(f'\n{model_name} - Iteration {iteration + 1}/{n_iterations}')
        
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42+iteration)
        fold_results = []
        
        for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
            X_train_fold, X_val_fold = X_train[train_idx], X_train[val_idx]
            y_train_fold, y_val_fold = y_train[train_idx], y_train[val_idx]
            
            clf = get_classifier(model_name)
            clf.fit(X_train_fold, y_train_fold)
            
            y_pred = clf.predict(X_val_fold)
            y_pred_proba = clf.predict_proba(X_val_fold)
            
            metrics = calculate_metrics(y_val_fold, y_pred, y_pred_proba, classes)
            fold_results.append(metrics)
            
            print(f'  Fold {fold_idx + 1}/{n_splits} completed')
        
        all_iteration_results.append(fold_results)
    
    return all_iteration_results


def evaluate_on_test(X_train, y_train, X_test, y_test, model_name, n_iterations=5):
    classes = np.unique(y_train)
    iteration_results = []
    
    for iteration in range(n_iterations):
        clf = get_classifier(model_name)
        clf.fit(X_train, y_train)
        
        y_pred = clf.predict(X_test)
        y_pred_proba = clf.predict_proba(X_test)
        
        metrics = calculate_metrics(y_test, y_pred, y_pred_proba, classes)
        iteration_results.append(metrics)
        
        print(f'  {model_name} - Test Iteration {iteration + 1}/{n_iterations} completed')
    
    return iteration_results


def compute_statistics(all_iteration_results):
    metric_names = ['AUC', 'Acc', 'Pre', 'Rec', 'F1', 'MCC', 'Kappa', 'MSE', 'MAE']
    
    all_values = {metric: [] for metric in metric_names}
    
    for iteration_results in all_iteration_results:
        for fold_result in iteration_results:
            for metric in metric_names:
                all_values[metric].append(fold_result[metric])
    
    stats = {}
    for metric in metric_names:
        stats[metric] = {
            'average': np.mean(all_values[metric]),
            'max': np.max(all_values[metric]),
            'std': np.std(all_values[metric])
        }
    
    return stats


def compute_statistics_test(iteration_results):
    metric_names = ['AUC', 'Acc', 'Pre', 'Rec', 'F1', 'MCC', 'Kappa', 'MSE', 'MAE']
    
    all_values = {metric: [] for metric in metric_names}
    
    for result in iteration_results:
        for metric in metric_names:
            all_values[metric].append(result[metric])
    
    stats = {}
    for metric in metric_names:
        stats[metric] = {
            'average': np.mean(all_values[metric]),
            'max': np.max(all_values[metric]),
            'std': np.std(all_values[metric])
        }
    
    return stats


if __name__ == '__main__':
    train_path = '/Users/akter/fahim/codegraph/traintry1.csv'
    test_path = '/Users/akter/fahim/codegraph/testtry1.csv'
    
    print('Loading data...')
    X_train, y_train, X_test, y_test = load_data(train_path, test_path)
    
    print(f'\nDataset Information:')
    print(f'  Training samples: {X_train.shape[0]}')
    print(f'  Test samples: {X_test.shape[0]}')
    print(f'  Features: {X_train.shape[1]}')
    print(f'  Classes: {len(np.unique(y_train))}')
    
    unique, counts = np.unique(y_train, return_counts=True)
    print(f'\nTraining Label Distribution:')
    for label, count in zip(unique, counts):
        print(f'  Class {label}: {count}')
    
    unique, counts = np.unique(y_test, return_counts=True)
    print(f'\nTest Label Distribution:')
    for label, count in zip(unique, counts):
        print(f'  Class {label}: {count}')
    
    models = ['Random Forest', 'Gradient Boosting', 'XGBoost', 'SVM', 
              'Logistic Regression', 'MLP']
    
    all_model_cv_results = {}
    all_model_test_results = {}
    
    for model_name in models:
        print(f'\n{"="*80}')
        print(f'Processing {model_name} - Cross-Validation on Training Set')
        print(f'{"="*80}')
        
        cv_results = cross_validate_on_train(X_train, y_train, model_name, n_splits=10, n_iterations=5)
        cv_stats = compute_statistics(cv_results)
        all_model_cv_results[model_name] = cv_stats
        
        print(f'\n{"="*80}')
        print(f'Processing {model_name} - Evaluation on Test Set')
        print(f'{"="*80}')
        
        test_results = evaluate_on_test(X_train, y_train, X_test, y_test, model_name, n_iterations=5)
        test_stats = compute_statistics_test(test_results)
        all_model_test_results[model_name] = test_stats
    
    print('\n\n' + '='*120)
    print(f'{"CROSS-VALIDATION RESULTS (Training Set) - AVERAGE VALUES":^120}')
    print('='*120)
    print(f'{"Model":<25} {"AUC":<10} {"Acc":<10} {"Pre":<10} {"Rec":<10} {"F1":<10} {"MCC":<10} {"Kappa":<10} {"MSE":<10} {"MAE":<10}')
    print('-'*120)
    
    for model_name in models:
        stats = all_model_cv_results[model_name]
        print(f'{model_name:<25} {stats["AUC"]["average"]:<10.4f} {stats["Acc"]["average"]:<10.4f} '
              f'{stats["Pre"]["average"]:<10.4f} {stats["Rec"]["average"]:<10.4f} {stats["F1"]["average"]:<10.4f} '
              f'{stats["MCC"]["average"]:<10.4f} {stats["Kappa"]["average"]:<10.4f} {stats["MSE"]["average"]:<10.4f} '
              f'{stats["MAE"]["average"]:<10.4f}')
    
    print('='*120)
    
    print('\n\n' + '='*120)
    print(f'{"CROSS-VALIDATION RESULTS (Training Set) - MAX VALUES":^120}')
    print('='*120)
    print(f'{"Model":<25} {"AUC":<10} {"Acc":<10} {"Pre":<10} {"Rec":<10} {"F1":<10} {"MCC":<10} {"Kappa":<10} {"MSE":<10} {"MAE":<10}')
    print('-'*120)
    
    for model_name in models:
        stats = all_model_cv_results[model_name]
        print(f'{model_name:<25} {stats["AUC"]["max"]:<10.4f} {stats["Acc"]["max"]:<10.4f} '
              f'{stats["Pre"]["max"]:<10.4f} {stats["Rec"]["max"]:<10.4f} {stats["F1"]["max"]:<10.4f} '
              f'{stats["MCC"]["max"]:<10.4f} {stats["Kappa"]["max"]:<10.4f} {stats["MSE"]["max"]:<10.4f} '
              f'{stats["MAE"]["max"]:<10.4f}')
    
    print('='*120)
    
    print('\n\n' + '='*120)
    print(f'{"CROSS-VALIDATION RESULTS (Training Set) - STANDARD DEVIATION":^120}')
    print('='*120)
    print(f'{"Model":<25} {"AUC":<10} {"Acc":<10} {"Pre":<10} {"Rec":<10} {"F1":<10} {"MCC":<10} {"Kappa":<10} {"MSE":<10} {"MAE":<10}')
    print('-'*120)
    
    for model_name in models:
        stats = all_model_cv_results[model_name]
        print(f'{model_name:<25} {stats["AUC"]["std"]:<10.4f} {stats["Acc"]["std"]:<10.4f} '
              f'{stats["Pre"]["std"]:<10.4f} {stats["Rec"]["std"]:<10.4f} {stats["F1"]["std"]:<10.4f} '
              f'{stats["MCC"]["std"]:<10.4f} {stats["Kappa"]["std"]:<10.4f} {stats["MSE"]["std"]:<10.4f} '
              f'{stats["MAE"]["std"]:<10.4f}')
    
    print('='*120)
    
    print('\n\n' + '='*120)
    print(f'{"TEST SET RESULTS - AVERAGE VALUES (5 Iterations)":^120}')
    print('='*120)
    print(f'{"Model":<25} {"AUC":<10} {"Acc":<10} {"Pre":<10} {"Rec":<10} {"F1":<10} {"MCC":<10} {"Kappa":<10} {"MSE":<10} {"MAE":<10}')
    print('-'*120)
    
    for model_name in models:
        stats = all_model_test_results[model_name]
        print(f'{model_name:<25} {stats["AUC"]["average"]:<10.4f} {stats["Acc"]["average"]:<10.4f} '
              f'{stats["Pre"]["average"]:<10.4f} {stats["Rec"]["average"]:<10.4f} {stats["F1"]["average"]:<10.4f} '
              f'{stats["MCC"]["average"]:<10.4f} {stats["Kappa"]["average"]:<10.4f} {stats["MSE"]["average"]:<10.4f} '
              f'{stats["MAE"]["average"]:<10.4f}')
    
    print('='*120)
    
    print('\n\n' + '='*120)
    print(f'{"TEST SET RESULTS - MAX VALUES (5 Iterations)":^120}')
    print('='*120)
    print(f'{"Model":<25} {"AUC":<10} {"Acc":<10} {"Pre":<10} {"Rec":<10} {"F1":<10} {"MCC":<10} {"Kappa":<10} {"MSE":<10} {"MAE":<10}')
    print('-'*120)
    
    for model_name in models:
        stats = all_model_test_results[model_name]
        print(f'{model_name:<25} {stats["AUC"]["max"]:<10.4f} {stats["Acc"]["max"]:<10.4f} '
              f'{stats["Pre"]["max"]:<10.4f} {stats["Rec"]["max"]:<10.4f} {stats["F1"]["max"]:<10.4f} '
              f'{stats["MCC"]["max"]:<10.4f} {stats["Kappa"]["max"]:<10.4f} {stats["MSE"]["max"]:<10.4f} '
              f'{stats["MAE"]["max"]:<10.4f}')
    
    print('='*120)
    
    print('\n\n' + '='*120)
    print(f'{"TEST SET RESULTS - STANDARD DEVIATION (5 Iterations)":^120}')
    print('='*120)
    print(f'{"Model":<25} {"AUC":<10} {"Acc":<10} {"Pre":<10} {"Rec":<10} {"F1":<10} {"MCC":<10} {"Kappa":<10} {"MSE":<10} {"MAE":<10}')
    print('-'*120)
    
    for model_name in models:
        stats = all_model_test_results[model_name]
        print(f'{model_name:<25} {stats["AUC"]["std"]:<10.4f} {stats["Acc"]["std"]:<10.4f} '
              f'{stats["Pre"]["std"]:<10.4f} {stats["Rec"]["std"]:<10.4f} {stats["F1"]["std"]:<10.4f} '
              f'{stats["MCC"]["std"]:<10.4f} {stats["Kappa"]["std"]:<10.4f} {stats["MSE"]["std"]:<10.4f} '
              f'{stats["MAE"]["std"]:<10.4f}')
    
    print('='*120)


# LIME inline error with codegraphnet embed

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import RobertaTokenizer, RobertaModel
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
import pandas as pd
import numpy as np
import ast
import re
import lime
import lime.lime_text
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

class ICFGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_icfg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            func_calls = {}
            func_defs = {}
            
            def traverse(node, parent_id=None):
                nonlocal node_id
                current_id = node_id
                node_type = type(node).__name__
                nodes.append(node_type)
                node_id += 1
                
                if parent_id is not None:
                    edges.append([parent_id, current_id])
                
                if isinstance(node, ast.FunctionDef):
                    func_defs[node.name] = current_id
                
                if isinstance(node, ast.Call):
                    if isinstance(node.func, ast.Name):
                        func_name = node.func.id
                        if func_name not in func_calls:
                            func_calls[func_name] = []
                        func_calls[func_name].append(current_id)
                
                for child in ast.iter_child_nodes(node):
                    traverse(child, current_id)
            
            traverse(tree)
            
            for func_name, call_sites in func_calls.items():
                if func_name in func_defs:
                    def_id = func_defs[func_name]
                    for call_id in call_sites:
                        edges.append([call_id, def_id])
                        edges.append([def_id, call_id])
            
            if len(nodes) == 0:
                nodes = ['Module']
                edges = []
            
            return nodes, edges
        except:
            return ['Module'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            with torch.no_grad():
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class DFGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_dfg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            var_last_write = {}
            var_last_read = {}
            
            def extract_dataflow(node, current_id):
                if isinstance(node, ast.Name):
                    var_name = node.id
                    
                    if isinstance(node.ctx, ast.Store):
                        if var_name in var_last_read:
                            for read_id in var_last_read[var_name]:
                                edges.append([read_id, current_id])
                        var_last_write[var_name] = current_id
                        var_last_read[var_name] = []
                    
                    elif isinstance(node.ctx, ast.Load):
                        if var_name in var_last_write:
                            edges.append([var_last_write[var_name], current_id])
                        if var_name not in var_last_read:
                            var_last_read[var_name] = []
                        var_last_read[var_name].append(current_id)
                
                for child in ast.iter_child_nodes(node):
                    extract_dataflow(child, current_id)
            
            def traverse(node):
                nonlocal node_id
                current_id = node_id
                nodes.append(type(node).__name__)
                node_id += 1
                extract_dataflow(node, current_id)
                for child in ast.iter_child_nodes(node):
                    traverse(child)
            
            traverse(tree)
            
            if len(nodes) == 0:
                nodes = ['Program']
                edges = []
            
            return nodes, edges
        except:
            return ['Program'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            with torch.no_grad():
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class CDGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_cdg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            control_stack = []
            
            def traverse(node, parent_id=None):
                nonlocal node_id
                current_id = node_id
                node_type = type(node).__name__
                nodes.append(node_type)
                node_id += 1
                
                if parent_id is not None:
                    edges.append([parent_id, current_id])
                
                if isinstance(node, (ast.If, ast.For, ast.While, ast.Try)):
                    for ctrl_id in control_stack:
                        edges.append([ctrl_id, current_id])
                    control_stack.append(current_id)
                    
                    for child in ast.iter_child_nodes(node):
                        traverse(child, current_id)
                    
                    control_stack.pop()
                else:
                    if control_stack:
                        for ctrl_id in control_stack:
                            edges.append([ctrl_id, current_id])
                    
                    for child in ast.iter_child_nodes(node):
                        traverse(child, current_id)
            
            traverse(tree)
            
            if len(nodes) == 0:
                nodes = ['Program']
                edges = []
            
            return nodes, edges
        except:
            return ['Program'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            with torch.no_grad():
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class StableGraphEncoder(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=384, output_dim=512):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(0.2)
        self.residual = nn.Linear(input_dim, output_dim)
    
    def forward(self, x, edge_index):
        identity = self.residual(x)
        x = self.conv1(x, edge_index)
        x = self.norm1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        x = self.norm2(x)
        x = x + identity
        return x


class HierarchicalGraphFusion(nn.Module):
    def __init__(self, graph_dim=512, num_heads=8):
        super().__init__()
        self.graph_dim = graph_dim
        self.icfg_encoder = StableGraphEncoder(768, 384, graph_dim)
        self.dfg_encoder = StableGraphEncoder(768, 384, graph_dim)
        self.cdg_encoder = StableGraphEncoder(768, 384, graph_dim)
        self.graph_attention = nn.MultiheadAttention(graph_dim, num_heads, dropout=0.1, batch_first=True)
        self.graph_norm = nn.LayerNorm(graph_dim)
        self.gate = nn.Sequential(
            nn.Linear(graph_dim * 3, graph_dim),
            nn.Sigmoid()
        )
        self.fusion = nn.Sequential(
            nn.Linear(graph_dim * 3, graph_dim * 2),
            nn.LayerNorm(graph_dim * 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(graph_dim * 2, graph_dim)
        )
        
    def forward(self, icfg_data, dfg_data, cdg_data):
        h_icfg = self.icfg_encoder(icfg_data.x, icfg_data.edge_index)
        h_dfg = self.dfg_encoder(dfg_data.x, dfg_data.edge_index)
        h_cdg = self.cdg_encoder(cdg_data.x, cdg_data.edge_index)
        icfg_global = torch.mean(h_icfg, dim=0, keepdim=True)
        dfg_global = torch.mean(h_dfg, dim=0, keepdim=True)
        cdg_global = torch.mean(h_cdg, dim=0, keepdim=True)
        graph_stack = torch.stack([icfg_global, dfg_global, cdg_global], dim=1)
        attended, _ = self.graph_attention(graph_stack, graph_stack, graph_stack)
        attended = self.graph_norm(attended + graph_stack)
        fused = torch.cat([attended[:, 0], attended[:, 1], attended[:, 2]], dim=-1)
        gate_weights = self.gate(fused)
        output = self.fusion(fused)
        output = output * gate_weights
        return output.squeeze(0)


class ImprovedUnifiedModel(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.tokenizer = RobertaTokenizer.from_pretrained('microsoft/graphcodebert-base')
        self.graphcodebert = RobertaModel.from_pretrained('microsoft/graphcodebert-base')
        for param in self.graphcodebert.parameters():
            param.requires_grad = True
        self.icfg_builder = ICFGBuilder(self.graphcodebert)
        self.dfg_builder = DFGBuilder(self.graphcodebert)
        self.cdg_builder = CDGBuilder(self.graphcodebert)
        self.graph_fusion = HierarchicalGraphFusion(graph_dim=512, num_heads=8)
        self.code_projection = nn.Linear(768, 512)
        self.multimodal_fusion = nn.Sequential(
            nn.Linear(512 + 512, 768),
            nn.LayerNorm(768),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(768, 512),
            nn.LayerNorm(512)
        )
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        self.criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
        
    def build_graph_data(self, code, device):
        icfg_nodes, icfg_edges = self.icfg_builder.build_icfg(code)
        dfg_nodes, dfg_edges = self.dfg_builder.build_dfg(code)
        cdg_nodes, cdg_edges = self.cdg_builder.build_cdg(code)
        icfg_x = self.icfg_builder.nodes_to_embeddings(icfg_nodes, self.tokenizer, device)
        dfg_x = self.dfg_builder.nodes_to_embeddings(dfg_nodes, self.tokenizer, device)
        cdg_x = self.cdg_builder.nodes_to_embeddings(cdg_nodes, self.tokenizer, device)
        if len(icfg_edges) == 0:
            icfg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
        else:
            icfg_edge_index = torch.tensor(icfg_edges, dtype=torch.long, device=device).t().contiguous()
        if len(dfg_edges) == 0:
            dfg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
        else:
            dfg_edge_index = torch.tensor(dfg_edges, dtype=torch.long, device=device).t().contiguous()
        if len(cdg_edges) == 0:
            cdg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
        else:
            cdg_edge_index = torch.tensor(cdg_edges, dtype=torch.long, device=device).t().contiguous()
        icfg_data = Data(x=icfg_x, edge_index=icfg_edge_index)
        dfg_data = Data(x=dfg_x, edge_index=dfg_edge_index)
        cdg_data = Data(x=cdg_x, edge_index=cdg_edge_index)
        return icfg_data, dfg_data, cdg_data
        
    def forward(self, code):
        device = next(self.parameters()).device
        icfg_data, dfg_data, cdg_data = self.build_graph_data(code, device)
        graph_repr = self.graph_fusion(icfg_data, dfg_data, cdg_data)
        tokens = self.tokenizer(
            code, 
            return_tensors='pt', 
            truncation=True, 
            max_length=512, 
            padding='max_length'
        )
        tokens = {k: v.to(device) for k, v in tokens.items()}
        code_output = self.graphcodebert(**tokens)
        code_repr = code_output.last_hidden_state[:, 0, :]
        code_repr = self.code_projection(code_repr)
        combined = torch.cat([graph_repr.unsqueeze(0), code_repr], dim=-1)
        fused_repr = self.multimodal_fusion(combined)
        logits = self.classifier(fused_repr)
        return logits


def remo(code):
    code = re.sub(r'/\*.*?\*/', '', code, flags=re.DOTALL)
    code = re.sub(r'//.*?$', '', code, flags=re.MULTILINE)
    code = re.sub(r'^\s*[\n\r]', '', code, flags=re.MULTILINE)
    return code.strip()


def predict_proba(texts):
    predictions = []
    for text in texts:
        logits = model(text)
        probs = F.softmax(logits, dim=-1)
        predictions.append(probs.detach().cpu().numpy()[0])
    return np.array(predictions)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Device: {device}')

model = ImprovedUnifiedModel(num_classes=6).to(device)
model.load_state_dict(torch.load('/Users/akter/fahim/codegraph/final_model1.pt', map_location=device, weights_only=True))
model.eval()

f = pd.read_csv('/Users/akter/fahim/data/testpro.csv')
f1 = f[['func', 'label']]

i = 4897
class_names = ['non-vul', 'CWE-119', 'CWE-120', 'CWE-469', 'CWE-476', 'CWE-other']

cleaned_code = remo(f1['func'][i])

with torch.no_grad():
    logits = model(cleaned_code)
    predicted_probs = F.softmax(logits, dim=-1).cpu().numpy()[0]
    predicted_label = np.argmax(predicted_probs)

print(f"Predicted Label: {class_names[predicted_label]}")
print(f"Prediction Probabilities: {predicted_probs}")

explainer = lime.lime_text.LimeTextExplainer(bow=False, class_names=class_names)
exp = explainer.explain_instance(cleaned_code, predict_proba, num_features=8, num_samples=200, labels=[predicted_label])
exp.show_in_notebook(predict_proba=False, show_predicted_value=False)

html_file = "vulnerability_lime_explanation.html"
exp.save_to_file(html_file)
print(f"Explanation saved to {html_file}")

# ROC-PR curve

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import RobertaTokenizer, RobertaModel
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, Batch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from tqdm import tqdm
import warnings
import ast
warnings.filterwarnings('ignore')

import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"


class ICFGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_icfg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            func_calls = {}
            func_defs = {}
            
            def traverse(node, parent_id=None):
                nonlocal node_id
                current_id = node_id
                node_type = type(node).__name__
                nodes.append(node_type)
                node_id += 1
                
                if parent_id is not None:
                    edges.append([parent_id, current_id])
                
                if isinstance(node, ast.FunctionDef):
                    func_defs[node.name] = current_id
                
                if isinstance(node, ast.Call):
                    if isinstance(node.func, ast.Name):
                        func_name = node.func.id
                        if func_name not in func_calls:
                            func_calls[func_name] = []
                        func_calls[func_name].append(current_id)
                
                for child in ast.iter_child_nodes(node):
                    traverse(child, current_id)
            
            traverse(tree)
            
            for func_name, call_sites in func_calls.items():
                if func_name in func_defs:
                    def_id = func_defs[func_name]
                    for call_id in call_sites:
                        edges.append([call_id, def_id])
                        edges.append([def_id, call_id])
            
            if len(nodes) == 0:
                nodes = ['Module']
                edges = []
            
            return nodes, edges
        except:
            return ['Module'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            with torch.no_grad():
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class DFGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_dfg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            var_last_write = {}
            var_last_read = {}
            
            def extract_dataflow(node, current_id):
                if isinstance(node, ast.Name):
                    var_name = node.id
                    
                    if isinstance(node.ctx, ast.Store):
                        if var_name in var_last_read:
                            for read_id in var_last_read[var_name]:
                                edges.append([read_id, current_id])
                        var_last_write[var_name] = current_id
                        var_last_read[var_name] = []
                    
                    elif isinstance(node.ctx, ast.Load):
                        if var_name in var_last_write:
                            edges.append([var_last_write[var_name], current_id])
                        if var_name not in var_last_read:
                            var_last_read[var_name] = []
                        var_last_read[var_name].append(current_id)
                
                for child in ast.iter_child_nodes(node):
                    extract_dataflow(child, current_id)
            
            def traverse(node):
                nonlocal node_id
                current_id = node_id
                nodes.append(type(node).__name__)
                node_id += 1
                extract_dataflow(node, current_id)
                for child in ast.iter_child_nodes(node):
                    traverse(child)
            
            traverse(tree)
            
            if len(nodes) == 0:
                nodes = ['Program']
                edges = []
            
            return nodes, edges
        except:
            return ['Program'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            with torch.no_grad():
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class CDGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_cdg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            control_stack = []
            
            def traverse(node, parent_id=None):
                nonlocal node_id
                current_id = node_id
                node_type = type(node).__name__
                nodes.append(node_type)
                node_id += 1
                
                if parent_id is not None:
                    edges.append([parent_id, current_id])
                
                if isinstance(node, (ast.If, ast.For, ast.While, ast.Try)):
                    for ctrl_id in control_stack:
                        edges.append([ctrl_id, current_id])
                    control_stack.append(current_id)
                    
                    for child in ast.iter_child_nodes(node):
                        traverse(child, current_id)
                    
                    control_stack.pop()
                else:
                    if control_stack:
                        for ctrl_id in control_stack:
                            edges.append([ctrl_id, current_id])
                    
                    for child in ast.iter_child_nodes(node):
                        traverse(child, current_id)
            
            traverse(tree)
            
            if len(nodes) == 0:
                nodes = ['Program']
                edges = []
            
            return nodes, edges
        except:
            return ['Program'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            with torch.no_grad():
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class StableGraphEncoder(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=384, output_dim=512):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(0.2)
        self.residual = nn.Linear(input_dim, output_dim)
    
    def forward(self, x, edge_index):
        identity = self.residual(x)
        
        x = self.conv1(x, edge_index)
        x = self.norm1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        
        x = self.conv2(x, edge_index)
        x = self.norm2(x)
        
        x = x + identity
        return x


class HierarchicalGraphFusion(nn.Module):
    def __init__(self, graph_dim=512, num_heads=8):
        super().__init__()
        self.graph_dim = graph_dim
        
        self.icfg_encoder = StableGraphEncoder(768, 384, graph_dim)
        self.dfg_encoder = StableGraphEncoder(768, 384, graph_dim)
        self.cdg_encoder = StableGraphEncoder(768, 384, graph_dim)
        
        self.graph_attention = nn.MultiheadAttention(graph_dim, num_heads, dropout=0.1, batch_first=True)
        self.graph_norm = nn.LayerNorm(graph_dim)
        
        self.gate = nn.Sequential(
            nn.Linear(graph_dim * 3, graph_dim),
            nn.Sigmoid()
        )
        
        self.fusion = nn.Sequential(
            nn.Linear(graph_dim * 3, graph_dim * 2),
            nn.LayerNorm(graph_dim * 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(graph_dim * 2, graph_dim)
        )
        
    def forward(self, icfg_data, dfg_data, cdg_data):
        h_icfg = self.icfg_encoder(icfg_data.x, icfg_data.edge_index)
        h_dfg = self.dfg_encoder(dfg_data.x, dfg_data.edge_index)
        h_cdg = self.cdg_encoder(cdg_data.x, cdg_data.edge_index)
        
        icfg_global = torch.mean(h_icfg, dim=0, keepdim=True)
        dfg_global = torch.mean(h_dfg, dim=0, keepdim=True)
        cdg_global = torch.mean(h_cdg, dim=0, keepdim=True)
        
        graph_stack = torch.stack([icfg_global, dfg_global, cdg_global], dim=1)
        
        attended, _ = self.graph_attention(graph_stack, graph_stack, graph_stack)
        attended = self.graph_norm(attended + graph_stack)
        
        fused = torch.cat([attended[:, 0], attended[:, 1], attended[:, 2]], dim=-1)
        
        gate_weights = self.gate(fused)
        output = self.fusion(fused)
        output = output * gate_weights
        
        return output.squeeze(0)


class ImprovedUnifiedModel(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.tokenizer = RobertaTokenizer.from_pretrained('microsoft/graphcodebert-base')
        self.graphcodebert = RobertaModel.from_pretrained('microsoft/graphcodebert-base')
        
        for param in self.graphcodebert.parameters():
            param.requires_grad = True
        
        self.icfg_builder = ICFGBuilder(self.graphcodebert)
        self.dfg_builder = DFGBuilder(self.graphcodebert)
        self.cdg_builder = CDGBuilder(self.graphcodebert)
        
        self.graph_fusion = HierarchicalGraphFusion(graph_dim=512, num_heads=8)
        
        self.code_projection = nn.Linear(768, 512)
        
        self.multimodal_fusion = nn.Sequential(
            nn.Linear(512 + 512, 768),
            nn.LayerNorm(768),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(768, 512),
            nn.LayerNorm(512)
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
        self.criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
        
    def build_graph_data(self, code, device):
        icfg_nodes, icfg_edges = self.icfg_builder.build_icfg(code)
        dfg_nodes, dfg_edges = self.dfg_builder.build_dfg(code)
        cdg_nodes, cdg_edges = self.cdg_builder.build_cdg(code)
        
        icfg_x = self.icfg_builder.nodes_to_embeddings(icfg_nodes, self.tokenizer, device)
        dfg_x = self.dfg_builder.nodes_to_embeddings(dfg_nodes, self.tokenizer, device)
        cdg_x = self.cdg_builder.nodes_to_embeddings(cdg_nodes, self.tokenizer, device)
        
        if len(icfg_edges) == 0:
            icfg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
        else:
            icfg_edge_index = torch.tensor(icfg_edges, dtype=torch.long, device=device).t().contiguous()
        
        if len(dfg_edges) == 0:
            dfg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
        else:
            dfg_edge_index = torch.tensor(dfg_edges, dtype=torch.long, device=device).t().contiguous()
        
        if len(cdg_edges) == 0:
            cdg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
        else:
            cdg_edge_index = torch.tensor(cdg_edges, dtype=torch.long, device=device).t().contiguous()
        
        icfg_data = Data(x=icfg_x, edge_index=icfg_edge_index)
        dfg_data = Data(x=dfg_x, edge_index=dfg_edge_index)
        cdg_data = Data(x=cdg_x, edge_index=cdg_edge_index)
        
        return icfg_data, dfg_data, cdg_data
        
    def forward(self, code):
        device = next(self.parameters()).device
        
        icfg_data, dfg_data, cdg_data = self.build_graph_data(code, device)
        graph_repr = self.graph_fusion(icfg_data, dfg_data, cdg_data)
        
        tokens = self.tokenizer(
            code, 
            return_tensors='pt', 
            truncation=True, 
            max_length=512, 
            padding='max_length'
        )
        tokens = {k: v.to(device) for k, v in tokens.items()}
        
        code_output = self.graphcodebert(**tokens)
        code_repr = code_output.last_hidden_state[:, 0, :]
        code_repr = self.code_projection(code_repr)
        
        combined = torch.cat([graph_repr.unsqueeze(0), code_repr], dim=-1)
        fused_repr = self.multimodal_fusion(combined)
        
        logits = self.classifier(fused_repr)
        return logits


class CodeDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            'func': str(row['func']),
            'label': int(row['label'])
        }



plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['figure.dpi'] = 300


def load_trained_model(model_path, num_classes=6, device='mps'):
    model = ImprovedUnifiedModel(num_classes=num_classes).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()
    print(f"✓ Model loaded from {model_path}")
    return model


def predict_on_new_data(model, test_df, device='mps'):
    print(f"\n{'='*70}")
    print(f"Test Data Information:")
    print(f"  Shape: {test_df.shape}")
    print(f"  Columns: {test_df.columns.tolist()}")
    
    if 'label' in test_df.columns:
        print(f"\nLabel Distribution:")
        print(test_df['label'].value_counts().sort_index())
    
    test_dataset = CodeDataset(test_df)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False)
    
    predictions = []
    ground_truth = []
    probabilities = []
    
    model.eval()
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Analyzing test data'):
            code = batch['func'][0] if isinstance(batch['func'], list) else batch['func']
            
            logits = model(code)
            if logits.dim() == 1:
                logits = logits.unsqueeze(0)
            
            probs = torch.softmax(logits, dim=1)
            pred = torch.argmax(logits, dim=1).cpu().item()
            
            predictions.append(pred)
            probabilities.append(probs.cpu().numpy()[0])
            
            if 'label' in batch:
                label = batch['label']
                if isinstance(label, torch.Tensor):
                    if label.dim() == 0:
                        label = label.item()
                    else:
                        label = label[0].item() if len(label) > 0 else label.item()
                ground_truth.append(label)
    
    num_classes = len(probabilities[0])
    test_df['predicted_label'] = predictions
    for i in range(num_classes):
        test_df[f'prob_class_{i}'] = [p[i] for p in probabilities]
    
    return test_df, predictions, ground_truth, probabilities


def plot_roc_curves_per_class(ground_truth, probabilities, num_classes=6, save_path='roc_curves_per_class.png'):
    y_true_bin = label_binarize(ground_truth, classes=range(num_classes))
    probs_array = np.array(probabilities)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.ravel()
    
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']
    
    for i in range(num_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], probs_array[:, i])
        roc_auc = auc(fpr, tpr)
        
        axes[i].plot(fpr, tpr, color=colors[i], lw=2.5, 
                    label=f'ROC curve (AUC = {roc_auc:.3f})')
        axes[i].plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.5)
        axes[i].set_xlim([0.0, 1.0])
        axes[i].set_ylim([0.0, 1.05])
        axes[i].set_xlabel('False Positive Rate', fontweight='bold')
        axes[i].set_ylabel('True Positive Rate', fontweight='bold')
        axes[i].set_title(f'Class {i} ROC Curve', fontweight='bold', fontsize=13)
        axes[i].legend(loc="lower right", frameon=True, shadow=True)
        axes[i].grid(True, alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ ROC curves saved to {save_path}")
    plt.close()


def plot_pr_curves_per_class(ground_truth, probabilities, num_classes=6, save_path='pr_curves_per_class.png'):
    y_true_bin = label_binarize(ground_truth, classes=range(num_classes))
    probs_array = np.array(probabilities)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.ravel()
    
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']
    
    for i in range(num_classes):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], probs_array[:, i])
        avg_precision = average_precision_score(y_true_bin[:, i], probs_array[:, i])
        
        axes[i].plot(recall, precision, color=colors[i], lw=2.5,
                    label=f'PR curve (AP = {avg_precision:.3f})')
        axes[i].set_xlim([0.0, 1.0])
        axes[i].set_ylim([0.0, 1.05])
        axes[i].set_xlabel('Recall', fontweight='bold')
        axes[i].set_ylabel('Precision', fontweight='bold')
        axes[i].set_title(f'Class {i} Precision-Recall Curve', fontweight='bold', fontsize=13)
        axes[i].legend(loc="lower left", frameon=True, shadow=True)
        axes[i].grid(True, alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ PR curves saved to {save_path}")
    plt.close()


def plot_combined_roc_curves(ground_truth, probabilities, num_classes=6, save_path='roc_curves_combined.png'):
    y_true_bin = label_binarize(ground_truth, classes=range(num_classes))
    probs_array = np.array(probabilities)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']
    
    for i in range(num_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], probs_array[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=colors[i], lw=2.5, 
               label=f'Class {i} (AUC = {roc_auc:.3f})')
    
    ax.plot([0, 1], [0, 1], 'k--', lw=2, alpha=0.5, label='Random Classifier')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontweight='bold', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontweight='bold', fontsize=12)
    ax.set_title('Multi-Class ROC Curves', fontweight='bold', fontsize=14)
    ax.legend(loc="lower right", frameon=True, shadow=True, fontsize=10)
    ax.grid(True, alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Combined ROC curves saved to {save_path}")
    plt.close()


def plot_combined_pr_curves(ground_truth, probabilities, num_classes=6, save_path='pr_curves_combined.png'):
    y_true_bin = label_binarize(ground_truth, classes=range(num_classes))
    probs_array = np.array(probabilities)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']
    
    for i in range(num_classes):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], probs_array[:, i])
        avg_precision = average_precision_score(y_true_bin[:, i], probs_array[:, i])
        ax.plot(recall, precision, color=colors[i], lw=2.5,
               label=f'Class {i} (AP = {avg_precision:.3f})')
    
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('Recall', fontweight='bold', fontsize=12)
    ax.set_ylabel('Precision', fontweight='bold', fontsize=12)
    ax.set_title('Multi-Class Precision-Recall Curves', fontweight='bold', fontsize=14)
    ax.legend(loc="lower left", frameon=True, shadow=True, fontsize=10)
    ax.grid(True, alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Combined PR curves saved to {save_path}")
    plt.close()


def plot_confusion_matrix(ground_truth, predictions, num_classes=6, save_path='confusion_matrix.png'):
    cm = confusion_matrix(ground_truth, predictions)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', square=True, 
                cbar_kws={'label': 'Count'}, ax=ax1, linewidths=1, linecolor='white')
    ax1.set_xlabel('Predicted Label', fontweight='bold', fontsize=12)
    ax1.set_ylabel('True Label', fontweight='bold', fontsize=12)
    ax1.set_title('Confusion Matrix (Counts)', fontweight='bold', fontsize=14)
    
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='RdYlGn', square=True,
                cbar_kws={'label': 'Proportion'}, ax=ax2, linewidths=1, linecolor='white')
    ax2.set_xlabel('Predicted Label', fontweight='bold', fontsize=12)
    ax2.set_ylabel('True Label', fontweight='bold', fontsize=12)
    ax2.set_title('Confusion Matrix (Normalized)', fontweight='bold', fontsize=14)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Confusion matrix saved to {save_path}")
    plt.close()


def plot_class_wise_metrics(ground_truth, predictions, num_classes=6, save_path='class_wise_metrics.png'):
    precision, recall, f1, support = precision_recall_fscore_support(
        ground_truth, predictions, labels=range(num_classes), zero_division=0
    )
    
    x = np.arange(num_classes)
    width = 0.25
    
    fig, ax = plt.subplots(figsize=(12, 7))
    
    bars1 = ax.bar(x - width, precision, width, label='Precision', color='#3498db', edgecolor='black', linewidth=1.2)
    bars2 = ax.bar(x, recall, width, label='Recall', color='#2ecc71', edgecolor='black', linewidth=1.2)
    bars3 = ax.bar(x + width, f1, width, label='F1-Score', color='#e74c3c', edgecolor='black', linewidth=1.2)
    
    ax.set_xlabel('Class', fontweight='bold', fontsize=12)
    ax.set_ylabel('Score', fontweight='bold', fontsize=12)
    ax.set_title('Class-wise Performance Metrics', fontweight='bold', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels([f'Class {i}' for i in range(num_classes)])
    ax.legend(frameon=True, shadow=True, fontsize=11)
    ax.grid(True, alpha=0.3, linestyle='--', axis='y')
    ax.set_ylim([0, 1.1])
    
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Class-wise metrics saved to {save_path}")
    plt.close()


def plot_class_distribution(ground_truth, predictions, num_classes=6, save_path='class_distribution.png'):
    true_counts = pd.Series(ground_truth).value_counts().sort_index()
    pred_counts = pd.Series(predictions).value_counts().sort_index()
    
    all_classes = range(num_classes)
    true_counts = true_counts.reindex(all_classes, fill_value=0)
    pred_counts = pred_counts.reindex(all_classes, fill_value=0)
    
    x = np.arange(num_classes)
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(12, 7))
    
    bars1 = ax.bar(x - width/2, true_counts.values, width, label='True Labels', 
                   color='#3498db', edgecolor='black', linewidth=1.2)
    bars2 = ax.bar(x + width/2, pred_counts.values, width, label='Predicted Labels',
                   color='#e74c3c', edgecolor='black', linewidth=1.2)
    
    ax.set_xlabel('Class', fontweight='bold', fontsize=12)
    ax.set_ylabel('Count', fontweight='bold', fontsize=12)
    ax.set_title('True vs Predicted Class Distribution', fontweight='bold', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels([f'Class {i}' for i in range(num_classes)])
    ax.legend(frameon=True, shadow=True, fontsize=11)
    ax.grid(True, alpha=0.3, linestyle='--', axis='y')
    
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                   f'{int(height)}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Class distribution saved to {save_path}")
    plt.close()


def plot_metrics_summary(metrics, save_path='metrics_summary.png'):
    metric_names = list(metrics.keys())
    metric_values = list(metrics.values())
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    colors = plt.cm.viridis(np.linspace(0, 1, len(metric_names)))
    bars = ax.barh(metric_names, metric_values, color=colors, edgecolor='black', linewidth=1.2)
    
    ax.set_xlabel('Score', fontweight='bold', fontsize=12)
    ax.set_title('Overall Model Performance Metrics', fontweight='bold', fontsize=14)
    ax.grid(True, alpha=0.3, linestyle='--', axis='x')
    
    for i, (bar, value) in enumerate(zip(bars, metric_values)):
        ax.text(value + 0.01, bar.get_y() + bar.get_height()/2,
               f'{value:.4f}', ha='left', va='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Metrics summary saved to {save_path}")
    plt.close()


def plot_prediction_confidence(probabilities, predictions, ground_truth, save_path='prediction_confidence.png'):
    probs_array = np.array(probabilities)
    max_probs = np.max(probs_array, axis=1)
    
    correct = np.array(predictions) == np.array(ground_truth)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    ax1.hist(max_probs[correct], bins=50, alpha=0.7, label='Correct', color='#2ecc71', edgecolor='black')
    ax1.hist(max_probs[~correct], bins=50, alpha=0.7, label='Incorrect', color='#e74c3c', edgecolor='black')
    ax1.set_xlabel('Maximum Probability', fontweight='bold', fontsize=12)
    ax1.set_ylabel('Frequency', fontweight='bold', fontsize=12)
    ax1.set_title('Prediction Confidence Distribution', fontweight='bold', fontsize=14)
    ax1.legend(frameon=True, shadow=True)
    ax1.grid(True, alpha=0.3, linestyle='--', axis='y')
    
    confidence_bins = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    accuracies = []
    bin_labels = []
    
    for i in range(len(confidence_bins)-1):
        mask = (max_probs >= confidence_bins[i]) & (max_probs < confidence_bins[i+1])
        if mask.sum() > 0:
            acc = correct[mask].mean()
            accuracies.append(acc)
            bin_labels.append(f'{confidence_bins[i]:.1f}-{confidence_bins[i+1]:.1f}')
    
    bars = ax2.bar(range(len(accuracies)), accuracies, color='#3498db', edgecolor='black', linewidth=1.2)
    ax2.set_xlabel('Confidence Range', fontweight='bold', fontsize=12)
    ax2.set_ylabel('Accuracy', fontweight='bold', fontsize=12)
    ax2.set_title('Accuracy vs Confidence', fontweight='bold', fontsize=14)
    ax2.set_xticks(range(len(bin_labels)))
    ax2.set_xticklabels(bin_labels, rotation=45)
    ax2.grid(True, alpha=0.3, linestyle='--', axis='y')
    ax2.set_ylim([0, 1.1])
    
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.02,
               f'{height:.3f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Prediction confidence saved to {save_path}")
    plt.close()


def calculate_comprehensive_metrics(ground_truth, predictions, probabilities):
    accuracy = accuracy_score(ground_truth, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        ground_truth, predictions, average='weighted', zero_division=0
    )
    
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        ground_truth, predictions, average='macro', zero_division=0
    )
    
    try:
        probs_array = np.array(probabilities)
        auc_score = roc_auc_score(ground_truth, probs_array, multi_class='ovr', average='weighted')
    except:
        auc_score = 0.0
        print("Warning: Could not calculate AUC")
    
    mcc = matthews_corrcoef(ground_truth, predictions)
    kappa = cohen_kappa_score(ground_truth, predictions)
    mse = mean_squared_error(ground_truth, predictions)
    mae = mean_absolute_error(ground_truth, predictions)
    
    metrics = {
        'AUC': auc_score,
        'Accuracy': accuracy,
        'Precision_Weighted': precision,
        'Recall_Weighted': recall,
        'F1_Weighted': f1,
        'Precision_Macro': precision_macro,
        'Recall_Macro': recall_macro,
        'F1_Macro': f1_macro,
        'MCC': mcc,
        'Kappa': kappa,
        'MSE': mse,
        'MAE': mae
    }
    
    print(f"\n{'='*70}")
    print(f"{'COMPREHENSIVE TEST RESULTS':^70}")
    print(f"{'='*70}")
    print(f"\n{'Performance Metrics:'}")
    for key, value in metrics.items():
        print(f"  {key:20s}: {value:.4f}")
    print(f"{'='*70}\n")
    
    print("Detailed Classification Report:")
    print(classification_report(ground_truth, predictions, zero_division=0))
    
    return metrics


if __name__ == '__main__':
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f'Device: {device}')

    MODEL_PATH = '/Users/akter/fahim/codegraph/final_model1.pt'
    TEST_CSV_PATH = '/Users/akter/fahim/data/testpro.csv'
    NUM_CLASSES = 6
    
    df1 = pd.read_csv(TEST_CSV_PATH)
    train_label0_sample = df1[df1['label'] == 0].sample(n=900, random_state=42)
    train_others = df1[df1['label'] != 0]
    test_df = pd.concat([train_label0_sample, train_others], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
    
    model = load_trained_model(MODEL_PATH, num_classes=NUM_CLASSES, device=device)
    
    results_df, predictions, ground_truth, probabilities = predict_on_new_data(
        model, test_df, device=device
    )
    
    metrics = calculate_comprehensive_metrics(ground_truth, predictions, probabilities)
    
    print("\n" + "="*70)
    print("GENERATING PUBLICATION-QUALITY FIGURES")
    print("="*70 + "\n")
    
    plot_roc_curves_per_class(ground_truth, probabilities, NUM_CLASSES, 'figure_roc_per_class.png')
    plot_pr_curves_per_class(ground_truth, probabilities, NUM_CLASSES, 'figure_pr_per_class.png')
    plot_combined_roc_curves(ground_truth, probabilities, NUM_CLASSES, 'figure_roc_combined.png')
    plot_combined_pr_curves(ground_truth, probabilities, NUM_CLASSES, 'figure_pr_combined.png')
    plot_confusion_matrix(ground_truth, predictions, NUM_CLASSES, 'figure_confusion_matrix.png')
    plot_class_wise_metrics(ground_truth, predictions, NUM_CLASSES, 'figure_class_metrics.png')
    plot_class_distribution(ground_truth, predictions, NUM_CLASSES, 'figure_class_distribution.png')
    plot_metrics_summary(metrics, 'figure_metrics_summary.png')
    plot_prediction_confidence(probabilities, predictions, ground_truth, 'figure_prediction_confidence.png')
    
    output_path = 'test_results_with_predictions.csv'
    results_df.to_csv(output_path, index=False)
    print(f"\n✓ Results saved to {output_path}")
    
    metrics_df = pd.DataFrame([metrics])
    metrics_output_path = 'test_metrics_summary.csv'
    metrics_df.to_csv(metrics_output_path, index=False)
    print(f"✓ Metrics summary saved to {metrics_output_path}")
    
    misclassified = results_df[results_df['label'] != results_df['predicted_label']]
    print(f"\nMisclassified samples: {len(misclassified)}")
    if len(misclassified) > 0:
        misclassified.to_csv('misclassified_samples.csv', index=False)
        print(f"✓ Misclassified samples saved to misclassified_samples.csv")
    
    print("\n" + "="*70)
    print("ALL FIGURES GENERATED SUCCESSFULLY!")
    print("="*70)